# Evaluation Results Viewer
Reads one or more `eval_results.json` files (produced by `tools/test_tracked.py`) and presents the data as pandas DataFrames.

In [ ]:
import hashlib
import json
import os
import re
import subprocess
import tempfile

import matplotlib.pyplot as plt
import pandas as pd
from matplotlib.lines import Line2D

# On-screen figures and saved PDFs are the same LaTeX-typeset plot.
PLOT_PDF_DIR = '/local/home/nkoefarago/mmpose/benchmark/plots'
plt.rcParams.update({
    'text.usetex': True,
    'font.family': 'serif',
    'font.serif': ['Computer Modern Roman'],
    'axes.unicode_minus': False,
    'text.latex.preamble': r'\usepackage{amsmath}',
})


def latex_text(text) -> str:
    """Escape plain text for usetex. Segments already in ``$...$`` are kept."""
    raw = '' if text is None else str(text)
    parts = re.split(r'(\$[^$]*\$)', raw)

    def _escape(part: str) -> str:
        part = part.replace('\\', '\x00')
        part = (
            part.replace('&', r'\&')
            .replace('%', r'\%')
            .replace('$', r'\$')
            .replace('#', r'\#')
            .replace('_', r'\_')
            .replace('{', r'\{')
            .replace('}', r'\}')
            .replace('~', r'\textasciitilde{}')
            .replace('^', r'\textasciicircum{}')
            .replace('\u2014', '---')
            .replace('\u2013', '--')
        )
        return part.replace('\x00', r'\textbackslash{}')

    return ''.join(part if part.startswith('$') else _escape(part) for part in parts)


def show_and_save_plot(fig, name: str, *, bbox_inches=None):
    """Write *fig* to a LaTeX PDF and display that saved file."""
    from IPython.display import Image, display

    os.makedirs(PLOT_PDF_DIR, exist_ok=True)
    path = os.path.join(PLOT_PDF_DIR, f'{name}.pdf')
    fig.savefig(path, format='pdf', bbox_inches=bbox_inches)
    plt.close(fig)

    with tempfile.NamedTemporaryFile(suffix='.png', delete=False) as tmp:
        preview = tmp.name
    subprocess.run(
        [
            'gs', '-q', '-dSAFER', '-dBATCH', '-dNOPAUSE',
            '-sDEVICE=png16m', '-r160', f'-sOutputFile={preview}', path,
        ],
        check=True,
    )
    print(f'Saved {path}')
    display(Image(filename=preview))

# ── Configuration ────────────────────────────────────────────────────────────
# One path or a list of paths. Each file has the same nested JSON structure and
# holds either raw runs or runs from exactly one post-processing strategy. Name
# post-processed files `..._post_processed_<strategy>.json` (e.g.
# `..._post_processed_smoothnet_ws8.json`) so each strategy gets its own marker
# shape when plotted alongside the raw results and any other strategies.
RESULTS_FILES = [
    # "/local/home/nkoefarago/mmpose/benchmark/results/20260610_coco_e2e.json",
    # "/local/home/nkoefarago/mmpose/benchmark/results/20260610_coco_topdown.json",
    # "/local/home/nkoefarago/mmpose/benchmark/results/20260815_coco_e2e.json",

    # "/local/home/nkoefarago/mmpose/benchmark/results/20260610_crowdpose_e2e.json",
    # "/local/home/nkoefarago/mmpose/benchmark/results/20260610_crowdpose_topdown.json",
    # "/local/home/nkoefarago/mmpose/benchmark/results/20260815_crowdpose_e2e.json",

    # "/local/home/nkoefarago/mmpose/benchmark/results/20260611_ochuman_e2e.json",
    # "/local/home/nkoefarago/mmpose/benchmark/results/20260611_ochuman_topdown.json",
    # "/local/home/nkoefarago/mmpose/benchmark/results/20261018_ochuman_e2e.json",

    # "/local/home/nkoefarago/mmpose/benchmark/results/20260715_3dpw_e2e.json",
    # "/local/home/nkoefarago/mmpose/benchmark/results/20260715_3dpw_topdown.json",

    "/local/home/nkoefarago/mmpose/benchmark/results/20260715_emdb_e2e.json",
    "/local/home/nkoefarago/mmpose/benchmark/results/20260715_emdb_topdown.json",
    "/local/home/nkoefarago/mmpose/benchmark/results/20260815_emdb_e2e.json",
    "/local/home/nkoefarago/mmpose/benchmark/results/20260806_emdb_video.json",
    "/local/home/nkoefarago/mmpose/benchmark/results/20260816_emdb_tracking.json",

    "/local/home/nkoefarago/mmpose/benchmark/results/20261026_emdb_postproc_ema.json",
    "/local/home/nkoefarago/mmpose/benchmark/results/20261026_emdb_postproc_oneeuro.json",
    "/local/home/nkoefarago/mmpose/benchmark/results/20261026_emdb_postproc_smoothnetw8.json",
    "/local/home/nkoefarago/mmpose/benchmark/results/20261026_emdb_postproc_smoothnetw16.json",
    # "/local/home/nkoefarago/mmpose/benchmark/results/20261026_emdb_postproc_smoothnetw32.json",

    "/local/home/nkoefarago/mmpose/benchmark/results/20261104_emdb_postproc_gpkalman.json",

    # "/local/home/nkoefarago/mmpose/benchmark/results/20261028_posetrack21_e2e.json",
    # "/local/home/nkoefarago/mmpose/benchmark/results/20261028_posetrack21_topdown.json",
    # "/local/home/nkoefarago/mmpose/benchmark/results/20261029_posetrack21_tracking.json",
    # "/local/home/nkoefarago/mmpose/benchmark/results/20261029_posetrack21_video.json",
    # "/local/home/nkoefarago/mmpose/benchmark/results/20261102_posetrack21_pose_tracking.json",
    # "/local/home/nkoefarago/mmpose/benchmark/results/20261105_posetrack21_pose_tracking.json",
    # "/local/home/nkoefarago/mmpose/benchmark/results/20261106_posetrack21_PCT-huge-rfdetr_tracking.json",

    # "/local/home/nkoefarago/mmpose/benchmark/results/20261025_coco_detector_sweep.json",

    # "/local/home/nkoefarago/mmpose/benchmark/results/20261102_posetrack21_pose_tracking.json",
    # "/local/home/nkoefarago/mmpose/benchmark/results/20261105_posetrack21_pose_tracking.json",
]

if isinstance(RESULTS_FILES, str):
    RESULTS_FILES = [RESULTS_FILES]

# ── Post-processing strategy per file ───────────────────────────────────────
# Each results file holds either raw runs or runs from exactly one post-processing
# strategy. The strategy name is used to pick a distinct marker shape per file, so
# any number of post-processing strategies (each in its own file) can be overlaid
# on the same plot alongside the raw (non post-processed) results.
RAW_STRATEGY = 'raw'
DEFAULT_MARKER = 'o'
# Cycled (in order of first appearance) across post-processing strategies found in
# the loaded data; extend if you load more than 9 distinct strategies at once.
STRATEGY_MARKER_POOL = ['2', '1', 'x', '+', '*', 'v', 'D', 's', '8']

# Legend text for each post-processing strategy. A key matches the strategy id
# exactly, or as a suffix after '_' (so 'ema' matches
# '20261026_emdb_postproc_ema'). Edit the values to change the post-processing
# legend; the order of entries here is the legend order. Ids with no match are
# shown unchanged, after the entries below. Segments in ``$...$`` are passed
# through to LaTeX.
STRATEGY_DISPLAY_NAMES = {
    'raw': 'raw',
    'ema': 'EMA',
    'oneeuro': 'One Euro',
    'smoothnetw8': r'SmoothNet ($w{=}8$)',
    'smoothnetw16': r'SmoothNet ($w{=}16$)',
    'smoothnetw32': r'SmoothNet ($w{=}32$)',
    'gpkalman': 'GP Kalman SORT',
}


def _strategy_display_key(strategy: str):
    """``STRATEGY_DISPLAY_NAMES`` key matching *strategy*, or None."""
    if strategy in STRATEGY_DISPLAY_NAMES:
        return strategy
    matches = [
        key for key in STRATEGY_DISPLAY_NAMES
        if key and strategy.endswith('_' + key)
    ]
    if not matches:
        return None
    return max(matches, key=len)


def strategy_display_name(strategy: str) -> str:
    """Legend label for a post-processing strategy id."""
    key = _strategy_display_key(strategy)
    if key is None:
        return strategy
    return STRATEGY_DISPLAY_NAMES[key]


def strategy_legend_sort_key(strategy: str):
    """Legend order: ``STRATEGY_DISPLAY_NAMES`` order, then unmatched ids."""
    key = _strategy_display_key(strategy)
    if key is None:
        return (1, strategy)
    return (0, list(STRATEGY_DISPLAY_NAMES).index(key), strategy)


def infer_strategy_label(path: str, is_post_processed: bool) -> str:
    """Post-processing strategy name for a results file (used for marker + legend).

    Every results file is assumed to hold either raw runs or runs from exactly one
    post-processing strategy. Prefers the convention
    `..._post_processed_<strategy>.json` -> 'strategy'; otherwise falls back to the
    filename stem so every distinct post-processed file still gets its own marker.
    """
    if not is_post_processed:
        return RAW_STRATEGY
    stem = os.path.splitext(os.path.basename(path))[0]
    tag = '_post_processed_'
    if tag in stem:
        return stem.split(tag, 1)[1]
    if stem.startswith('results_'):
        stem = stem[len('results_'):]
    return stem


def load_eval_results(path):
    with open(path) as f:
        raw = json.load(f)
    records = []
    for model_name, variants in raw.items():
        for variant, runs in variants.items():
            for run in runs:
                post_processed = bool(run.get('post_processed', False))
                record = {
                    'model': model_name,
                    'variant': variant,
                    'timestamp': pd.Timestamp(run['timestamp']),
                    'config': run.get('config', ''),
                    'checkpoint': run.get('checkpoint', ''),
                    'post_processed': post_processed,
                    'post_config': run.get('post_config') or '',
                    'strategy': infer_strategy_label(path, post_processed),
                    'source_file': path,
                }
                record.update(run.get('metrics', {}))
                records.append(record)
    return records


# ── Load & flatten ───────────────────────────────────────────────────────────
records = []
for path in RESULTS_FILES:
    if not os.path.isfile(path):
        raise FileNotFoundError(f'Results file not found: {path}')
    records.extend(load_eval_results(path))

df = pd.DataFrame(records)

# Derive all metric columns (everything after the fixed columns)
ID_COLS = ["model", "variant", "timestamp"]
META_COLS = ['config', 'checkpoint', 'post_processed', 'post_config', 'strategy', 'source_file']
METRIC_COLS = [c for c in df.columns if c not in (ID_COLS + META_COLS)]


def strategy_sort_key(strategy: str):
    """Sort key placing 'raw' first, then strategies alphabetically."""
    return (strategy != RAW_STRATEGY, strategy)


def _build_strategy_markers(strategies):
    markers = {RAW_STRATEGY: DEFAULT_MARKER}
    others = sorted({s for s in strategies if s != RAW_STRATEGY})
    for i, strategy in enumerate(others):
        markers[strategy] = STRATEGY_MARKER_POOL[i % len(STRATEGY_MARKER_POOL)]
    return markers


# Fixed marker per strategy across all plots (order of first appearance in the
# full loaded dataset), so the same strategy always renders with the same shape.
STRATEGY_MARKERS = _build_strategy_markers(df['strategy'].unique()) if not df.empty else {}

if not df.empty and 'strategy' in df.columns:
    print('Post-processing legend labels (edit STRATEGY_DISPLAY_NAMES to change):')
    for _strategy in sorted(df['strategy'].dropna().unique(), key=strategy_legend_sort_key):
        print(f'  {_strategy}: {strategy_display_name(_strategy)}')


def marker_for_strategy(strategy: str) -> str:
    return STRATEGY_MARKERS.get(strategy, DEFAULT_MARKER)

# Fixed plot color per known model family. tab20 alone only has 20 colors, so
# tab20b/tab20c are concatenated (60 total); extra HSV hues if that is still short.
# First 20 families keep their original tab20 colors. Extend order when adding families.
_MODEL_COLOR_ORDER = [
    'DARK', 'HRFormer', 'HRNet', 'MSPN', 'PCT', 'PETR', 'RF-DETR-Pose', 'RSN',
    'RTMPose', 'Sapiens', 'SimCC', 'UDP', 'ViTPose', 'YOLO-Pose', 'YOLO26-Pose',
    'Poseidon', 'TAR-ViTPose', 'PAVE-Net', 'DETRPose', 'GroupPose', 'QueryPose',
    'QueryPose-light', 'ED-Pose', 'AlphaPose', 'OpenPifPaf',
]


def _qualitative_palette(n):
    colors = []
    for name in ('tab20', 'tab20b', 'tab20c'):
        colors.extend(plt.get_cmap(name).colors)
    if len(colors) < n:
        extra = n - len(colors)
        colors.extend(plt.get_cmap('hsv')((i + 0.5) / extra) for i in range(extra))
    return list(colors[:n])


# Extra slots so unknown families hash into leftover qualitative colors, not
# back onto a reserved family.
_FALLBACK_SLOTS = 20
_model_palette = _qualitative_palette(len(_MODEL_COLOR_ORDER) + _FALLBACK_SLOTS)
MODEL_COLORS = {m: _model_palette[i] for i, m in enumerate(_MODEL_COLOR_ORDER)}

_TEMPORAL_PREFIXES = ('emdb', 'temporal', '3dpw', 'posetrack21')


def metric_label(col: str) -> str:
    """Short label for a metric column (strip emdb/ or temporal/ prefix)."""
    for prefix in _TEMPORAL_PREFIXES:
        prefix_slash = f'{prefix}/'
        if col.startswith(prefix_slash):
            return col[len(prefix_slash):]
    return col


def resolve_temporal_metric_col(df, suffix: str, prefixes=_TEMPORAL_PREFIXES):
    """Return emdb/* or temporal/* column for a metric suffix, preferring populated columns."""
    candidates = [
        f'{prefix}/{suffix}' for prefix in prefixes
        if f'{prefix}/{suffix}' in df.columns
    ]
    if not candidates:
        return None
    populated = [c for c in candidates if df[c].notna().any()]
    return populated[0] if populated else candidates[0]


def resolve_temporal_metric_cols(df, suffixes, prefixes=_TEMPORAL_PREFIXES):
    """Resolve multiple temporal metric columns (emdb/* or temporal/*)."""
    cols = []
    for suffix in suffixes:
        col = resolve_temporal_metric_col(df, suffix, prefixes)
        if col is not None:
            cols.append(col)
    return cols


def color_for_model(model: str):
    """Fixed color for known families; stable hash fallback for others."""
    if model in MODEL_COLORS:
        return MODEL_COLORS[model]
    digest = hashlib.md5(model.encode()).hexdigest()
    n_reserved = len(_MODEL_COLOR_ORDER)
    n_fallback = len(_model_palette) - n_reserved
    idx = n_reserved + (int(digest, 16) % n_fallback)
    return _model_palette[idx]


def scatter_runs(ax, sub, x_col, y_col, model, *, show_model_label=False):
    """Scatter points for one model; marker shape encodes post-processing strategy."""
    color = color_for_model(model)
    strategies = sorted(sub['strategy'].dropna().unique(), key=strategy_sort_key)
    for i, strategy in enumerate(strategies):
        strategy_sub = sub[sub['strategy'] == strategy]
        if strategy_sub.empty:
            continue
        ax.scatter(
            strategy_sub[x_col],
            strategy_sub[y_col],
            s=18,
            alpha=0.85,
            color=color,
            marker=marker_for_strategy(strategy),
            label=latex_text(model) if (show_model_label and i == 0) else None,
        )


def strategy_shape_legend_handles(strategies):
    """Legend entries mapping marker shape to post-processing strategy."""
    handles = []
    for strategy in sorted(set(strategies), key=strategy_legend_sort_key):
        handles.append(Line2D(
            [0], [0],
            marker=marker_for_strategy(strategy),
            color='0.45',
            linestyle='None',
            markersize=6 if strategy == RAW_STRATEGY else 9,
            label=latex_text(strategy_display_name(strategy)),
        ))
    return handles


def _match_legend_widths(*legends):
    """Give every legend the same frame width (the widest one).

    The width is stored in points and reapplied at draw time, so a later
    save at a different dpi still lines the frames up.
    """
    legends = [leg for leg in legends if leg is not None]
    if len(legends) < 2:
        return
    fig = legends[0].axes.figure
    renderer = fig.canvas.get_renderer()
    for leg in legends:
        leg._legend_box.set_width(None)
    widths = [leg.get_window_extent(renderer).width for leg in legends]
    target_pt = max(widths) / renderer.points_to_pixels(1.0)

    def _apply(leg, renderer):
        target_px = target_pt * renderer.points_to_pixels(1.0)
        pad = leg.borderpad * renderer.points_to_pixels(leg._fontsize)
        leg._legend_box.set_width(max(0.0, target_px - 2 * pad))

    for leg in legends:
        _apply(leg, renderer)
        original_draw = leg.draw

        def draw(renderer, _leg=leg, _orig=original_draw):
            _apply(_leg, renderer)
            return _orig(renderer)

        leg.draw = draw


def add_model_and_strategy_legends(ax, strategies):
    """Model legend (top) plus a separate post-processing marker legend (bottom).

    The marker-shape legend is omitted when only one strategy is plotted, since all
    markers look the same and need no explanation. When both are shown, their
    frames are stretched to the same width.
    """
    model_legend = ax.legend(
        title='model',
        bbox_to_anchor=(1.02, 1),
        loc='upper left',
        fontsize=8,
        framealpha=0.9,
    )
    if len(set(strategies)) > 1:
        ax.add_artist(model_legend)
        strategy_legend = ax.legend(
            handles=strategy_shape_legend_handles(strategies),
            title='post-processing',
            bbox_to_anchor=(1.02, 0),
            loc='lower left',
            fontsize=8,
            framealpha=0.9,
        )
        _match_legend_widths(model_legend, strategy_legend)


FPS_COL = 'perf/e2e/fps'
POSTPROC_TOTAL_COL = 'perf/postproc/total_s'
POSTPROC_FPS_COL = 'perf/postproc/fps'
POSTPROC_LATENCY_COL = 'perf/postproc/latency_ms_per_frame'
COMBINED_FPS_COL = 'perf/combined/fps'


def add_throughput_fps(data, include_postproc_time=False):
    """Copy *data* and add ``COMBINED_FPS_COL`` as plot throughput (FPS).

    Default (``include_postproc_time=False``): the base model's e2e FPS
    (``perf/e2e/fps``), including for post-processed runs — post-processing
    does not change the pose model's inference speed.

    With ``include_postproc_time=True``: combined model + post-processing
    FPS, ``num_frames / (num_frames / e2e_fps + postproc_total_s)``.
    ``num_frames`` is recovered as ``postproc_fps * postproc_total_s``
    (fallback: ``1000 * total_s / latency_ms``). Raw runs with no postproc
    time keep the base-model FPS.
    """
    out = data.copy()
    if not include_postproc_time:
        out[COMBINED_FPS_COL] = (
            out[FPS_COL] if FPS_COL in out.columns
            else pd.Series(index=out.index, dtype=float)
        )
        return out

    e2e_fps = (
        out[FPS_COL].astype(float) if FPS_COL in out.columns
        else pd.Series(index=out.index, dtype=float)
    )
    if POSTPROC_TOTAL_COL in out.columns:
        total_s = out[POSTPROC_TOTAL_COL].astype(float).fillna(0.0)
    else:
        total_s = pd.Series(0.0, index=out.index)

    n_frames = pd.Series(index=out.index, dtype=float)
    if POSTPROC_FPS_COL in out.columns:
        n_frames = (out[POSTPROC_FPS_COL].astype(float) * total_s).where(
            total_s > 0)
    if POSTPROC_LATENCY_COL in out.columns:
        lat_ms = out[POSTPROC_LATENCY_COL].astype(float)
        n_from_lat = (1000.0 * total_s / lat_ms).where(
            (total_s > 0) & (lat_ms > 0))
        n_frames = n_frames.fillna(n_from_lat)

    denom = (n_frames / e2e_fps.where(e2e_fps > 0)) + total_s
    combined = n_frames / denom.where(denom > 0)
    out[COMBINED_FPS_COL] = combined.where(total_s > 0, e2e_fps)
    return out


def latest_plot_runs(data):
    """Latest run per (model, variant, strategy) for scatter plots."""
    return (
        data.sort_values('timestamp')
        .groupby(['model', 'variant', 'strategy'], sort=False)
        .last()
        .reset_index()
    )


def filter_by_model_variants(data, pairs=None):
    """Keep rows whose (model, variant) is in *pairs*. None keeps everything."""
    if pairs is None:
        return data.reset_index(drop=True)
    allowed = set(pairs)
    mask = data.apply(lambda r: (r['model'], r['variant']) in allowed, axis=1)
    return data[mask].reset_index(drop=True)


def exclude_models(data, excluded=None):
    """Drop rows named in *excluded*.

    Each entry is a model family (``'YOLO-Pose'``) or a ``(model, variant)``
    pair. A bare string drops every variant of that model. None or an empty
    list keeps every row.
    """
    if isinstance(excluded, str):
        excluded = [excluded]
    if not excluded:
        return data.reset_index(drop=True)
    families = []
    pairs = []
    for item in excluded:
        if isinstance(item, str):
            families.append(item)
        elif isinstance(item, (tuple, list)) and len(item) == 2:
            pairs.append((item[0], item[1]))
        else:
            raise TypeError(
                'EXCLUDE_MODELS entries must be model names or '
                f'(model, variant) pairs, got {item!r}'
            )
    out = data
    if families:
        out = out[~out['model'].isin(families)]
    if pairs:
        pair_index = pd.MultiIndex.from_frame(out[['model', 'variant']])
        out = out[~pair_index.isin(pairs)]
    return out.reset_index(drop=True)


# Display names and config stems that mean the same post-processing strategy.
# 'pgpt' stays distinct from 'pgpt-geom'.
_POSTPROC_CANONICAL = {
    'bot-sort': 'botsort',
    'byte-track': 'bytetrack',
    'deep-oc-sort': 'deepocsort',
    'oc-sort': 'ocsort',
    'occlu-boost': 'occluboost',
    'one-euro': 'oneeuro',
    'pgpt-geometric': 'pgpt-geom',
    'smoothnet-w8': 'smoothnetw8',
    'smoothnet-w16': 'smoothnetw16',
    'smoothnet-w32': 'smoothnetw32',
}


def canonical_postproc_name(name) -> str:
    """Normalize a strategy id, display name, or post_config path."""
    if name is None:
        return ''
    try:
        if pd.isna(name):
            return ''
    except (TypeError, ValueError):
        pass
    text = str(name).strip()
    if not text or text.lower() in {'none', 'nan'}:
        return ''
    text = os.path.splitext(os.path.basename(text))[0]
    text = text.lower().replace('_', '-').replace(' ', '-')
    text = re.sub(r'-+', '-', text).strip('-')
    if text.startswith('boxmot-'):
        text = text[len('boxmot-'):]
    return _POSTPROC_CANONICAL.get(text, text)


def _strategy_matches_postproc(strategy, canon: str) -> bool:
    if strategy is None:
        return False
    try:
        if pd.isna(strategy):
            return False
    except (TypeError, ValueError):
        pass
    text = str(strategy).strip().lower()
    if not text:
        return False
    hyphen = text.replace('_', '-')
    if hyphen == canon or hyphen.endswith('-' + canon):
        return True
    compact = canon.replace('-', '')
    if text.endswith('_' + compact) or text.endswith('_' + canon):
        return True
    return canonical_postproc_name(strategy_display_name(str(strategy))) == canon


def _variant_has_postproc(variant, canon: str) -> bool:
    """True when *canon* is the trailing '-' token sequence of a run name."""
    if not canon:
        return False
    if variant is None:
        return False
    try:
        if pd.isna(variant):
            return False
    except (TypeError, ValueError):
        pass
    tokens = [tok for tok in str(variant).lower().replace('_', '-').split('-') if tok]
    want = canon.split('-')
    return len(tokens) >= len(want) and tokens[-len(want):] == want


def row_matches_postproc(row, canons) -> bool:
    """True when a run uses one of the canonical post-processing strategies."""
    strategy = row['strategy'] if 'strategy' in row.index else ''
    variant = row['variant'] if 'variant' in row.index else ''
    post = row['post_config'] if 'post_config' in row.index else ''
    post_canon = canonical_postproc_name(post)
    for canon in canons:
        if post_canon == canon:
            return True
        if _strategy_matches_postproc(strategy, canon):
            return True
        if _variant_has_postproc(variant, canon):
            return True
    return False


def exclude_postproc(data, excluded=None):
    """Drop rows that use one of the named post-processing strategies.

    Each entry matches a file strategy (``raw``, ``ema``, ``smoothnetw8``,
    ``oneeuro``) or the tracker stored in ``post_config`` / the variant name
    (``bytetrack``, ``pgpt``, ``pgpt-geom``, ``botsort``, ``deepocsort``,
    ``gp-kalman-sort``). Display names such as ``ByteTrack`` and ``PGPT geom``
    match too. ``pgpt`` does not drop ``pgpt-geom``. None or empty keeps
    every row.
    """
    if isinstance(excluded, str):
        excluded = [excluded]
    if not excluded:
        return data.reset_index(drop=True)
    canons = []
    for item in excluded:
        if not isinstance(item, str) or not str(item).strip():
            raise TypeError(
                'EXCLUDE_POSTPROC entries must be strategy names, '
                f'got {item!r}'
            )
        canon = canonical_postproc_name(item)
        if canon:
            canons.append(canon)
    if not canons:
        return data.reset_index(drop=True)
    canons = set(canons)
    mask = data.apply(lambda row: row_matches_postproc(row, canons), axis=1)
    return data.loc[~mask].reset_index(drop=True)


# Detector families appended to the variant (run name) by the benchmark
# scripts, e.g. ``small-rfdetr`` or ``large-rfdetr-medium``. Used only to
# list suffixes; DETECTOR_TYPES matches any token, including ones not listed
# here. Extend when a new family is added to scripts/benchmark_configs_detectors.csv.
_KNOWN_DETECTOR_FAMILIES = ('fasterrcnn', 'rfdetr', 'rtmdet', 'yolo26')


def run_name_has_detector(variant, detector) -> bool:
    """True when *detector* is its own '-' delimited token in a run name.

    ``rfdetr`` matches ``small-rfdetr`` and ``large-rfdetr-medium``.
    ``rfdetr-medium`` matches only the sized suffix. Matching is
    case-insensitive.
    """
    if variant is None or (isinstance(variant, float) and pd.isna(variant)):
        name = ''
    else:
        name = str(variant).lower()
    token = str(detector).strip().lower()
    if not token:
        return False
    return f'-{token}-' in f'-{name}-'


def filter_by_detector(data, detectors=None):
    """Keep rows for the requested detectors, plus end-to-end runs.

    *detectors* is a string or iterable of tokens such as ``'rfdetr'`` or
    ``'rtmdet-medium'``. A run is kept when its variant contains one of those
    tokens, or when the variant has no detector suffix (end-to-end models
    such as YOLO-Pose and DETRPose). Other detector families are dropped.
    None keeps every row.
    """
    if isinstance(detectors, str):
        detectors = [detectors]
    if not detectors:
        return data.reset_index(drop=True)
    tokens = [str(token).strip() for token in detectors if str(token).strip()]
    if not tokens:
        return data.reset_index(drop=True)
    has_token = data['variant'].apply(
        lambda variant: any(
            run_name_has_detector(variant, token) for token in tokens
        )
    )
    end_to_end = data['variant'].map(detector_suffix_from_run_name).isna()
    return data.loc[has_token | end_to_end].reset_index(drop=True)


def detector_suffix_from_run_name(variant):
    """Detector suffix in a run name, or None when the run has no detector.

    ``large-rfdetr-medium`` -> ``rfdetr-medium``,
    ``hrnetw48-li-rtmdet`` -> ``rtmdet``.
    """
    if variant is None or (isinstance(variant, float) and pd.isna(variant)):
        return None
    parts = str(variant).lower().split('-')
    families = set(_KNOWN_DETECTOR_FAMILIES)
    for i in range(len(parts) - 1, -1, -1):
        if parts[i] in families:
            return '-'.join(parts[i:])
    return None


def available_detector_suffixes(data):
    """Run counts per detector suffix found in the variant (run name)."""
    suffixes = (
        latest_plot_runs(data)['variant']
        .map(detector_suffix_from_run_name)
        .fillna('(no detector)')
    )
    return (
        suffixes.value_counts()
        .rename_axis('detector')
        .reset_index(name='runs')
        .sort_values('detector')
        .reset_index(drop=True)
    )


NO_DETECTOR_LABEL = '(no detector)'
# Legend text for each detector suffix parsed from the run name. Edit the
# values to change the detector legend; the order of entries here is the
# legend order. Suffixes with no match are shown unchanged, after the entries
# below. Segments in ``$...$`` are passed through to LaTeX.
DETECTOR_DISPLAY_NAMES = {
    NO_DETECTOR_LABEL: 'end-to-end',
    'fasterrcnn': 'Faster R-CNN',
    'fasterrcnn-r50': 'Faster R-CNN R50',
    'rfdetr': 'RF-DETR',
    'rfdetr-nano': 'RF-DETR nano',
    'rfdetr-small': 'RF-DETR small',
    'rfdetr-medium': 'RF-DETR medium',
    'rfdetr-large': 'RF-DETR large',
    'rfdetr-xlarge': 'RF-DETR xlarge',
    'rfdetr-2xlarge': 'RF-DETR 2xlarge',
    'rtmdet': 'RTMDet',
    'rtmdet-tiny': 'RTMDet tiny',
    'rtmdet-small': 'RTMDet small',
    'rtmdet-medium': 'RTMDet medium',
    'rtmdet-large': 'RTMDet large',
    'rtmdet-xlarge': 'RTMDet xlarge',
    'yolo26': 'YOLO26',
    'yolo26-nano': 'YOLO26 nano',
    'yolo26-small': 'YOLO26 small',
    'yolo26-medium': 'YOLO26 medium',
    'yolo26-large': 'YOLO26 large',
    'yolo26-xlarge': 'YOLO26 xlarge',
}
# 'o' is reserved for end-to-end runs that have no detector suffix.
DETECTOR_MARKER_POOL = [
    's', '^', 'v', '<', '>', 'D', 'P', 'X', '*', '+', '2', '8', 'd',
    (5, 1, 0), (6, 1, 0), (7, 1, 0), (4, 1, 45), (8, 0, 0),
]
_DETECTOR_SIZE_RANK = {
    'nano': 0,
    'tiny': 1,
    'small': 2,
    'medium': 3,
    'large': 4,
    'xlarge': 5,
    '2xlarge': 6,
    'r50': 7,
}


def detector_label_from_run_name(variant) -> str:
    """Detector shown on plots, or '(no detector)' for end-to-end runs."""
    suffix = detector_suffix_from_run_name(variant)
    return suffix if suffix else NO_DETECTOR_LABEL


def detector_display_name(label: str) -> str:
    """Legend label for a detector suffix."""
    return DETECTOR_DISPLAY_NAMES.get(label, label)


def detector_legend_sort_key(label: str):
    """Legend order: ``DETECTOR_DISPLAY_NAMES`` order, then unmatched suffixes."""
    if label in DETECTOR_DISPLAY_NAMES:
        return (0, list(DETECTOR_DISPLAY_NAMES).index(label))
    return (1, detector_sort_key(label))


def detector_sort_key(label: str):
    """Family, then size (nano < ... < 2xlarge), so legends stay grouped."""
    if label == NO_DETECTOR_LABEL:
        return ('', -1, '')
    family, _, size = str(label).partition('-')
    return (family, _DETECTOR_SIZE_RANK.get(size, 50), size)


def _build_detector_markers(labels):
    markers = {NO_DETECTOR_LABEL: DEFAULT_MARKER}
    others = sorted(
        {label for label in labels if label and label != NO_DETECTOR_LABEL},
        key=detector_sort_key,
    )
    for i, label in enumerate(others):
        markers[label] = DETECTOR_MARKER_POOL[i % len(DETECTOR_MARKER_POOL)]
    return markers


# Fixed marker per detector across the detector-shaped plots, so the same
# detector always renders with the same shape.
DETECTOR_MARKERS = (
    _build_detector_markers(
        df['variant'].map(detector_label_from_run_name).unique()
    )
    if (not df.empty and 'variant' in df.columns) else {}
)


def marker_for_detector(label: str):
    return DETECTOR_MARKERS.get(label, DEFAULT_MARKER)


if not df.empty and 'variant' in df.columns:
    print('Detector legend labels (edit DETECTOR_DISPLAY_NAMES to change):')
    for _label in sorted(
        set(df['variant'].map(detector_label_from_run_name)),
        key=detector_legend_sort_key,
    ):
        print(f'  {_label}: {detector_display_name(_label)}')


def scatter_runs_by_detector(ax, sub, x_col, y_col, model, *, show_model_label=False):
    """Scatter points for one model; marker shape encodes the detector."""
    color = color_for_model(model)
    labels = sorted(
        sub['variant'].map(detector_label_from_run_name).unique(),
        key=detector_sort_key,
    )
    labeled_model = False
    for label in labels:
        det_sub = sub[sub['variant'].map(detector_label_from_run_name) == label]
        if det_sub.empty:
            continue
        ax.scatter(
            det_sub[x_col],
            det_sub[y_col],
            s=24,
            alpha=0.85,
            color=color,
            marker=marker_for_detector(label),
            label=latex_text(model) if (show_model_label and not labeled_model) else None,
        )
        labeled_model = True


def detector_shape_legend_handles(labels):
    """Legend entries mapping marker shape to detector display name."""
    handles = []
    for label in sorted(set(labels), key=detector_legend_sort_key):
        handles.append(Line2D(
            [0], [0],
            marker=marker_for_detector(label),
            color='0.45',
            linestyle='None',
            markersize=8,
            label=latex_text(detector_display_name(label)),
        ))
    return handles


def add_model_and_detector_legends(ax, labels):
    """Model legend (top) plus a separate detector marker legend (bottom).

    The marker-shape legend is omitted when only one detector is plotted.
    When both are shown, their frames are stretched to the same width.
    """
    model_legend = ax.legend(
        title='model',
        bbox_to_anchor=(1.02, 1),
        loc='upper left',
        fontsize=8,
        framealpha=0.9,
    )
    if len(set(labels)) > 1:
        ax.add_artist(model_legend)
        detector_legend = ax.legend(
            handles=detector_shape_legend_handles(labels),
            title='detector',
            bbox_to_anchor=(1.02, 0),
            loc='lower left',
            fontsize=8,
            framealpha=0.9,
        )
        _match_legend_widths(model_legend, detector_legend)


def available_model_variant_pairs(data):
    """Distinct (model, variant) pairs in *data* (latest run per post_processed flag)."""
    return (
        latest_plot_runs(data)[['model', 'variant']]
        .drop_duplicates()
        .sort_values(['model', 'variant'])
        .reset_index(drop=True)
    )


def _axis_limit(lo, hi, *, padding, clamp_hi=None):
    span = hi - lo
    if span <= 0:
        span = abs(hi) or abs(lo) or 1.0
    lo_p = lo - span * padding
    hi_p = hi + span * padding
    if clamp_hi is not None:
        hi_p = min(clamp_hi, hi_p)
    return (max(0, lo_p), hi_p)


def compute_axis_limits_from_data(data, *, fps_col=FPS_COL, padding=0.05):
    """Compute shared axis limits from latest runs in *data*."""
    latest = latest_plot_runs(data)
    limits = {}
    if fps_col in latest.columns and latest[fps_col].notna().any():
        vals = latest[fps_col].dropna()
        limits['fps'] = _axis_limit(vals.min(), vals.max(), padding=padding)
    for col in ('coco/AP', 'coco/AR'):
        if col in latest.columns and latest[col].notna().any():
            vals = latest[col].dropna()
            limits[col] = _axis_limit(vals.min(), vals.max(), padding=padding, clamp_hi=1.0)
    for suffix in ('bMPJAE', 'bMPJVE', 'tMPJAE', 'tMPJVE'):
        col = resolve_temporal_metric_col(latest, suffix)
        if col is not None and latest[col].notna().any():
            vals = latest[col].dropna()
            limits[suffix] = _axis_limit(vals.min(), vals.max(), padding=padding)
    hota_col = 'tracking/HOTA'
    if hota_col in latest.columns and latest[hota_col].notna().any():
        vals = latest[hota_col].dropna()
        limits['HOTA'] = _axis_limit(
            vals.min(), vals.max(), padding=padding, clamp_hi=1.0)
    return limits


def build_plot_axis_limits(reference_data, manual_limits, *, padding=0.05):
    """Merge auto-computed limits from *reference_data* with manual overrides."""
    computed = compute_axis_limits_from_data(reference_data, padding=padding)
    keys = (
        'fps', 'coco/AP', 'coco/AR', 'bMPJAE', 'bMPJVE', 'tMPJAE', 'tMPJVE',
        'HOTA',
    )
    return {
        key: manual_limits.get(key)
        if manual_limits.get(key) is not None
        else computed.get(key)
        for key in keys
    }


def apply_plot_axis_limits(ax, x_col, y_col, limits, *, fps_col=FPS_COL):
    """Apply shared axis limits configured for scatter plots."""
    if limits.get('fps') and x_col == fps_col:
        ax.set_xlim(limits['fps'])
    for y_key in (y_col, metric_label(y_col), y_col.rsplit('/', 1)[-1]):
        if limits.get(y_key):
            ax.set_ylim(limits[y_key])
            break


def plot_ap_ar_vs_fps(data, limits, name, *, include_postproc_time=False):
    """AP and AR as side-by-side subplots with one shared legend.

    Each panel keeps its own y-limits. Model labels are attached only on the
    recall panel, so the legend is drawn once, to the right of that panel.
    """
    ap_col = 'coco/AP'
    ar_col = 'coco/AR'
    missing = [c for c in (ap_col, ar_col) if c not in data.columns]
    if missing:
        print(f'Missing columns for plot: {missing}')
        return
    plot_df = add_throughput_fps(
        latest_plot_runs(data),
        include_postproc_time=include_postproc_time,
    )
    x_col = COMBINED_FPS_COL
    plot_df = plot_df.dropna(subset=[ap_col, ar_col, x_col])
    if plot_df.empty:
        print(f'No rows with {ap_col}, {ar_col}, and throughput FPS.')
        return

    fig, (ax_ap, ax_ar) = plt.subplots(1, 2, figsize=(14, 5), sharex=True)

    for model in plot_df['model'].unique():
        sub = plot_df[plot_df['model'] == model]
        scatter_runs(ax_ap, sub, x_col, ap_col, model, show_model_label=False)
        scatter_runs(ax_ar, sub, x_col, ar_col, model, show_model_label=True)

    if include_postproc_time:
        xlabel = 'combined FPS (model + post-processing)'
        title_fps = 'combined model + post-processing throughput'
    else:
        xlabel = 'e2e FPS'
        title_fps = 'end-to-end throughput'
    ax_ap.set_xlabel('FPS' if not include_postproc_time else xlabel)
    ax_ap.set_ylabel('AP')
    ax_ap.set_title(f'Average precision vs {title_fps}')
    ax_ap.grid(True, alpha=0.3)
    ax_ar.set_xlabel(xlabel)
    ax_ar.set_ylabel('AR')
    ax_ar.set_title(f'Recall vs {title_fps}')
    ax_ar.grid(True, alpha=0.3)

    apply_limits = dict(limits)
    if include_postproc_time and plot_df[x_col].notna().any():
        vals = plot_df[x_col].dropna()
        apply_limits['fps'] = _axis_limit(
            vals.min(), vals.max(), padding=AXIS_PADDING)
    apply_plot_axis_limits(ax_ap, x_col, ap_col, apply_limits, fps_col=x_col)
    apply_plot_axis_limits(ax_ar, x_col, ar_col, apply_limits, fps_col=x_col)
    add_model_and_strategy_legends(ax_ar, plot_df['strategy'].unique())
    fig.tight_layout()
    show_and_save_plot(fig, name, bbox_inches='tight')


def plot_two_metrics_vs_fps(
    data, limits, name, left_metric, right_metric, *,
    include_postproc_time=False,
):
    """Two metrics as side-by-side subplots with one shared legend.

    Each name is a column or a temporal suffix such as ``bMPJVE``. Each panel
    keeps its own y-limits. The legend is drawn once, on the right panel.
    """
    def _col(metric):
        if metric in data.columns:
            return metric
        return resolve_temporal_metric_col(data, metric)

    left_col = _col(left_metric)
    right_col = _col(right_metric)
    missing = [
        metric for metric, col in (
            (left_metric, left_col), (right_metric, right_col),
        ) if col is None
    ]
    if missing:
        print(f'Missing columns for plot: {missing}')
        return
    plot_df = add_throughput_fps(
        latest_plot_runs(data),
        include_postproc_time=include_postproc_time,
    )
    x_col = COMBINED_FPS_COL
    plot_df = plot_df.dropna(subset=[left_col, right_col, x_col])
    if plot_df.empty:
        print(f'No rows with {left_col}, {right_col}, and throughput FPS.')
        return

    fig, (ax_left, ax_right) = plt.subplots(1, 2, figsize=(14, 5), sharex=True)
    for model in plot_df['model'].unique():
        sub = plot_df[plot_df['model'] == model]
        scatter_runs(
            ax_left, sub, x_col, left_col, model, show_model_label=False)
        scatter_runs(
            ax_right, sub, x_col, right_col, model, show_model_label=True)

    if include_postproc_time:
        xlabel = 'combined FPS (model + post-processing)'
        title_fps = 'combined model + post-processing throughput'
    else:
        xlabel = 'e2e FPS'
        title_fps = 'end-to-end throughput'
    ax_left.set_xlabel('FPS' if not include_postproc_time else xlabel)
    ax_left.set_ylabel(metric_label(left_col))
    ax_left.set_title(f'{metric_label(left_col)} vs {title_fps}')
    ax_left.grid(True, alpha=0.3)
    ax_right.set_xlabel(xlabel)
    ax_right.set_ylabel(metric_label(right_col))
    ax_right.set_title(f'{metric_label(right_col)} vs {title_fps}')
    ax_right.grid(True, alpha=0.3)

    apply_limits = dict(limits)
    if include_postproc_time and plot_df[x_col].notna().any():
        vals = plot_df[x_col].dropna()
        apply_limits['fps'] = _axis_limit(
            vals.min(), vals.max(), padding=AXIS_PADDING)
    apply_plot_axis_limits(
        ax_left, x_col, left_col, apply_limits, fps_col=x_col)
    apply_plot_axis_limits(
        ax_right, x_col, right_col, apply_limits, fps_col=x_col)
    add_model_and_strategy_legends(ax_right, plot_df['strategy'].unique())
    fig.tight_layout()
    show_and_save_plot(fig, name, bbox_inches='tight')


print(
    f'Loaded {len(df)} evaluation run(s) from {len(RESULTS_FILES)} file(s) '
    f'| metrics: {METRIC_COLS}'
)


## Plot filter & fixed axis limits

Define named groups as hand-picked `(model, variant)` lists, set `ACTIVE_GROUP`, and lock axis ranges so different groups share the same plot scale.

`EXCLUDE_MODELS` drops model families (`'YOLO-Pose'`) or individual `(model, variant)` runs. `EXCLUDE_POSTPROC` drops post-processing strategies: a file strategy (`raw`, `ema`, `smoothnetw8`, `oneeuro`) or a tracker named in `post_config` or the run name (`bytetrack`, `pgpt`, `pgpt-geom`, `botsort`, `deepocsort`, `gp-kalman-sort`). Display names such as `ByteTrack` and `PGPT geom` match too. `pgpt` does not drop `pgpt-geom`. `DETECTOR_TYPES` keeps runs whose variant contains a detector token from the run name (`rfdetr`, `rtmdet`, `yolo26`, `fasterrcnn`, or a sized suffix such as `rfdetr-medium`), and also keeps end-to-end runs that have no detector suffix. `rfdetr` keeps `small-rfdetr` and `large-rfdetr-medium` together with YOLO-Pose, DETRPose, and the other end-to-end models, and drops other detectors such as `rtmdet`. All three apply on top of `ACTIVE_GROUP`. Leave them empty to plot the whole group.

Re-run this cell after changing `MODEL_GROUPS`, `ACTIVE_GROUP`, `EXCLUDE_MODELS`, `EXCLUDE_POSTPROC`, `DETECTOR_TYPES`, or axis settings. Downstream plot cells use `df_plot` and `PLOT_AXIS_LIMITS`. The load cell defines the filter helpers, so re-run that cell once after pulling these changes.

In [ ]:
# ── Model groups (hand-defined) ───────────────────────────────────────────────
# Each group is a list of (model, variant) tuples. Set SHOW_AVAILABLE_PAIRS = True
# once to list every pair in the loaded data, then copy entries into MODEL_GROUPS.
SHOW_AVAILABLE_PAIRS = False

# Drop these on top of ACTIVE_GROUP. Each entry is a model family
# ('YOLO-Pose') or a (model, variant) pair. Empty keeps the whole group.
EXCLUDE_MODELS = ['Sapiens']

# Drop these post-processing strategies on top of ACTIVE_GROUP.
# A file strategy ('raw', 'ema', 'smoothnetw8', 'oneeuro') or a tracker in
# post_config / the run name ('bytetrack', 'pgpt', 'pgpt-geom', 'botsort',
# 'deepocsort', 'gp-kalman-sort'). Display names work too ('ByteTrack',
# 'PGPT geom', 'BoT-SORT'). 'pgpt' does not drop 'pgpt-geom'.
# Empty keeps every strategy. A string or a list both work.
EXCLUDE_POSTPROC = None

# Keep runs whose variant (run name) contains one of these detector tokens,
# and also keep end-to-end runs that have no detector suffix.
# Examples: 'rfdetr', 'rtmdet', 'yolo26', 'fasterrcnn', 'rfdetr-medium'.
# 'rfdetr' matches small-rfdetr and large-rfdetr-medium, keeps YOLO-Pose /
# DETRPose / other end-to-end models, and drops rtmdet (and any other
# detector). None keeps every run.
# A string or a list both work: 'rfdetr' or ['rfdetr', 'yolo26'].
DETECTOR_TYPES = 'rfdetr'

MODEL_GROUPS = {
    'light': [
        ('YOLO-Pose', 'large'),
        ('YOLO-Pose', 'medium'),
        ('YOLO-Pose', 'small'),
        ('YOLO-Pose', 'tiny'),
        ('YOLO26-Pose', 'xlarge'),
        ('YOLO26-Pose', 'large'),
        ('YOLO26-Pose', 'medium'),
        ('YOLO26-Pose', 'small'),
        ('YOLO26-Pose', 'nano'),
        ('RF-DETR-Pose', 'xlarge'),
        ('ViTPose', 'small-rfdetr'),
        ('UDP', 'hrnetw32-li-rtmdet'),
        ('UDP', 'cspnext-tiny-rfdetr'),
        ('RTMPose', 'medium-rfdetr'),
        ('RTMPose', 'small-rfdetr'),
        ('RTMPose', 'tiny-rfdetr'),
        ('SimCC', 'resnet50-si-rfdetr'),
        ('DETRPose', 'n'),
        ('DETRPose', 's'),
        ('DETRPose', 'm'),
        ('DETRPose', 'l'),
        ('DETRPose', 'x'),
    ],
    'heavy': [
        ('ViTPose', 'large-rfdetr'),
        ('ViTPose', 'huge-rfdetr'),
        ('UDP', 'hrnetw48-li-rtmdet'),
        ('SimCC', 'resnet50-li-rtmdet'),
        ('RTMPose', 'large-rtmdet'),
        ('PETR', 'swin-l'),
        ('PETR', 'resnet101'),
        ('PETR', 'resnet50'),
        ('PCT', 'base-rtmdet'),
        ('PCT', 'large-rtmdet'),
        ('PCT', 'huge-rtmdet'),
        ('MSPN', '4xresnet50-rtmdet'),
        ('HRNet', 'hrnetw48-si-rtmdet'),
        ('HRFormer', 'base-si-rtmdet'),
        ('DARK', 'hrnetw48-si-rtmdet'),
        ("Poseidon", "vits-posetrack21-rfdetr"),
        ("Poseidon", "vitb-posetrack21-rfdetr"),
        ("Poseidon", "vith-posetrack21-rfdetr"),
        ("TAR-ViTPose", "vitb-posetrack17-rfdetr"),
        ("TAR-ViTPose", "vith-posetrack17-rfdetr"),
        ("PAVE-Net", "r50-posetrack17"),
    ],
    "tracking": [
        ('PCT', 'huge_coco-rfdetr'),
        ('PCT', 'base-rfdetr'),
        ('TAR-ViTPose', 'vith-posetrack17-rfdetr'),
        ('ViTPose', 'large-rfdetr'),
        ('DETRPose', 'm'),
        ('DETRPose', 'l'),
        ('AlphaPose', 'fastpose-duc-res152-rtmdet'),
        ('OpenPifPaf', 'tshufflenetv2k30-trackingpose')
    ],
    "smoothing": [
        ('ViTPose', 'huge-rfdetr'),
        ('PCT', 'huge-rfdetr'),
        ('UDP', 'hrnetw48-li-rfdetr'),
        ('YOLO26-Pose', 'xlarge'),
        ('YOLO26-Pose', 'medium'),
        ('DETRPose', 'm'),
        ('DETRPose', 'l'),
        ('DETRPose', 'x'),
        ("TAR-ViTPose", "vith-posetrack17-rfdetr"),
        ('RTMPose', 'large-rfdetr'),
        ('YOLO-Pose', 'large'),
        ('YOLO-Pose', 'medium'),
    ]
}
ACTIVE_GROUP = 'smoothing'

# ── Fixed axis limits ────────────────────────────────────────────────────────
# Limits are computed from the reference dataset, then manual overrides win.
# Use AXIS_LIMITS_SOURCE='full' so filtered plots keep the same axes as the
# unfiltered baseline (recommended for cross-group comparison).
AXIS_LIMITS_SOURCE = 'full'  # 'full' | 'filtered'
AXIS_PADDING = 0.05
MANUAL_AXIS_LIMITS = {
    'fps': [0, 100],
    'coco/AP': None,
    'coco/AR': [0.52, 0.965],
    'bMPJAE': [0, 0.04],
    'bMPJVE': [0, 0.027],
    'tMPJAE': None,
    'tMPJVE': None,
    'HOTA': None,
}

if SHOW_AVAILABLE_PAIRS:
    display(
        available_model_variant_pairs(df)
        .style.set_caption('Available (model, variant) pairs — copy into MODEL_GROUPS')
    )
    display(
        available_detector_suffixes(df)
        .style.set_caption(
            'Detector suffixes in run names — use with DETECTOR_TYPES. '
            'A family token such as rfdetr also matches sized suffixes.'
        )
    )

_selected_pairs = MODEL_GROUPS.get(ACTIVE_GROUP) if ACTIVE_GROUP else None
if ACTIVE_GROUP and _selected_pairs is None:
    raise KeyError(
        f'ACTIVE_GROUP={ACTIVE_GROUP!r} not in MODEL_GROUPS. '
        f'Available: {sorted(MODEL_GROUPS)}'
    )

_exclude_items = (
    [EXCLUDE_MODELS] if isinstance(EXCLUDE_MODELS, str)
    else list(EXCLUDE_MODELS or [])
)
_loaded_models = set(df['model'])
_loaded_pairs = set(zip(df['model'], df['variant']))
_missing_exclude = []
for _item in _exclude_items:
    if isinstance(_item, str):
        if _item not in _loaded_models:
            _missing_exclude.append(_item)
    elif isinstance(_item, (tuple, list)) and len(_item) == 2:
        if (_item[0], _item[1]) not in _loaded_pairs:
            _missing_exclude.append(tuple(_item))
if _missing_exclude:
    print(
        'Warning: EXCLUDE_MODELS entries not found in loaded data:',
        _missing_exclude,
    )

if DETECTOR_TYPES:
    _det_tokens = (
        [DETECTOR_TYPES] if isinstance(DETECTOR_TYPES, str)
        else list(DETECTOR_TYPES)
    )
    _det_hit = df['variant'].apply(
        lambda variant: any(
            run_name_has_detector(variant, token) for token in _det_tokens
        )
    )
    if not _det_hit.any():
        print(
            'Warning: DETECTOR_TYPES matched no loaded runs. '
            'End-to-end runs are still included. '
            'Tokens are matched inside the variant (run name), '
            'e.g. rfdetr, rtmdet, yolo26, rfdetr-medium.'
        )

_postproc_items = (
    [EXCLUDE_POSTPROC] if isinstance(EXCLUDE_POSTPROC, str)
    else list(EXCLUDE_POSTPROC or [])
)
_missing_postproc = []
for _item in _postproc_items:
    _canon = canonical_postproc_name(_item)
    if not _canon or not df.apply(
        lambda row, canon=_canon: row_matches_postproc(row, {canon}),
        axis=1,
    ).any():
        _missing_postproc.append(_item)
if _missing_postproc:
    print(
        'Warning: EXCLUDE_POSTPROC entries not found in loaded data:',
        _missing_postproc,
    )

df_plot = filter_by_model_variants(df, _selected_pairs)
if ACTIVE_GROUP and _selected_pairs:
    _missing = set(_selected_pairs) - set(zip(df_plot['model'], df_plot['variant']))
    if _missing:
        print('Warning: group entries not found in loaded data:', sorted(_missing))

df_plot = exclude_models(df_plot, EXCLUDE_MODELS)
df_plot = filter_by_detector(df_plot, DETECTOR_TYPES)
df_plot = exclude_postproc(df_plot, EXCLUDE_POSTPROC)

_ref_for_limits = df if AXIS_LIMITS_SOURCE == 'full' else df_plot
PLOT_AXIS_LIMITS = build_plot_axis_limits(
    _ref_for_limits, MANUAL_AXIS_LIMITS, padding=AXIS_PADDING,
)

_filter_bits = [ACTIVE_GROUP or 'all']
if _exclude_items:
    _filter_bits.append('exclude ' + ', '.join(
        item if isinstance(item, str) else f'{item[0]}/{item[1]}'
        for item in _exclude_items
    ))
if DETECTOR_TYPES:
    _det_list = (
        [DETECTOR_TYPES] if isinstance(DETECTOR_TYPES, str)
        else list(DETECTOR_TYPES)
    )
    _filter_bits.append('detector ' + ', '.join(str(token) for token in _det_list))
if _postproc_items:
    _filter_bits.append(
        'exclude postproc ' + ', '.join(str(item) for item in _postproc_items)
    )
_group_label = ' | '.join(_filter_bits)
_latest_plot = latest_plot_runs(df_plot)
print(
    f'Plot filter: {_group_label} | '
    f'{len(df_plot)} run(s), {len(_latest_plot)} latest point(s) '
    f'| {df_plot["model"].nunique() if len(df_plot) else 0} model(s), '
    f'{df_plot.groupby(["model", "variant"]).ngroups if len(df_plot) else 0} '
    f'model-variant pair(s)'
)

# if df_plot.empty:
#     print('Warning: filter matched no runs — plots below will be empty.')
# else:
#     display(
#         _latest_plot[['model', 'variant', 'post_processed']]
#         .drop_duplicates()
#         .sort_values(['model', 'variant', 'post_processed'])
#         .reset_index(drop=True)
#         .style.set_caption(f'Included entries ({_group_label})')
#     )

_active_limits = {k: v for k, v in PLOT_AXIS_LIMITS.items() if v is not None}
print('Fixed axis limits:', _active_limits or '(auto per plot — set MANUAL_AXIS_LIMITS or load data)')

## AP vs e2e FPS

In [ ]:
AP_COL = 'coco/AP'
FPS_COL = 'perf/e2e/fps'
# Set to a positive int to label only the N highest-AP points (0 = no point labels).
ANNOTATE_TOP_N = 0

missing = [c for c in (AP_COL, FPS_COL) if c not in df_plot.columns]
if missing:
    print(f'Missing columns for plot: {missing}')
    print('Available metric columns:', METRIC_COLS)
else:
    plot_df = latest_plot_runs(df_plot).dropna(subset=[AP_COL, FPS_COL])

    if plot_df.empty:
        print(f'No rows with both {AP_COL} and {FPS_COL}.')
    else:
        models = plot_df['model'].unique()

        fig, ax = plt.subplots(figsize=(8, 5))
        for model in models:
            sub = plot_df[plot_df['model'] == model]
            scatter_runs(ax, sub, FPS_COL, AP_COL, model, show_model_label=True)

        if ANNOTATE_TOP_N > 0:
            top = plot_df.nlargest(ANNOTATE_TOP_N, AP_COL)
            for _, row in top.iterrows():
                ax.annotate(
                    latex_text(row['variant']),
                    (row[FPS_COL], row[AP_COL]),
                    xytext=(4, 4),
                    textcoords='offset points',
                    fontsize=7,
                    color=color_for_model(row['model']),
                )

        ax.set_xlabel('FPS')
        ax.set_ylabel('AP')
        ax.set_title('Average precision vs end-to-end throughput')
        ax.grid(True, alpha=0.3)
        apply_plot_axis_limits(ax, FPS_COL, AP_COL, PLOT_AXIS_LIMITS)
        add_model_and_strategy_legends(ax, plot_df['strategy'].unique())
        fig.tight_layout()

        # Variant names for each point (no overlap on the chart).
        # display(
        #     plot_df[['model', 'variant', 'post_processed', FPS_COL, AP_COL]]
        #     .sort_values([AP_COL, FPS_COL], ascending=[False, False])
        #     .reset_index(drop=True)
        #     .style.format({AP_COL: '{:.3f}', FPS_COL: '{:.1f}'})
        #     .set_caption('Points on plot (hover-free lookup)')
       
        # )
        show_and_save_plot(fig, 'ap_vs_fps')


## AR vs inference speed

Default x-axis is the **base model's** e2e FPS (`perf/e2e/fps`), including for post-processed runs. Set `INCLUDE_POSTPROC_TIME = True` to plot combined model + post-processing throughput instead: `num_frames / (num_frames / e2e_fps + postproc_total_s)`.

In [ ]:
AR_COL = 'coco/AR'
# False: x-axis is the base model's e2e FPS (perf/e2e/fps), including for
# post-processed runs. True: x-axis is combined model + post-processing FPS
# num_frames / (num_frames / e2e_fps + postproc_total_s); raw runs without
# postproc time are unchanged.
INCLUDE_POSTPROC_TIME = False
# Set to a positive int to label only the N highest-AR points (0 = no point labels).
ANNOTATE_TOP_N = 0

if AR_COL not in df_plot.columns:
    print(f'Missing column for plot: {AR_COL}')
    print('Available metric columns:', METRIC_COLS)
else:
    plot_df = add_throughput_fps(
        latest_plot_runs(df_plot),
        include_postproc_time=INCLUDE_POSTPROC_TIME,
    )
    x_col = COMBINED_FPS_COL
    plot_df = plot_df.dropna(subset=[AR_COL, x_col])

    if plot_df.empty:
        print(f'No rows with both {AR_COL} and throughput FPS.')
    else:
        models = plot_df['model'].unique()

        fig, ax = plt.subplots(figsize=(8, 5))
        for model in models:
            sub = plot_df[plot_df['model'] == model]
            scatter_runs(ax, sub, x_col, AR_COL, model, show_model_label=True)

        if ANNOTATE_TOP_N > 0:
            top = plot_df.nlargest(ANNOTATE_TOP_N, AR_COL)
            for _, row in top.iterrows():
                ax.annotate(
                    latex_text(row['variant']),
                    (row[x_col], row[AR_COL]),
                    xytext=(4, 4),
                    textcoords='offset points',
                    fontsize=7,
                    color=color_for_model(row['model']),
                )

        if INCLUDE_POSTPROC_TIME:
            xlabel = 'combined FPS (model + post-processing)'
            title_fps = 'combined model + post-processing throughput'
        else:
            xlabel = 'e2e FPS'
            title_fps = 'end-to-end throughput'

        ax.set_xlabel(xlabel)
        ax.set_ylabel('AR')
        ax.set_title(
            f'Recall vs {title_fps}'
        )
        ax.grid(True, alpha=0.3)

        limits = dict(PLOT_AXIS_LIMITS)
        if INCLUDE_POSTPROC_TIME and plot_df[x_col].notna().any():
            vals = plot_df[x_col].dropna()
            limits['fps'] = _axis_limit(
                vals.min(), vals.max(), padding=AXIS_PADDING)
        apply_plot_axis_limits(ax, x_col, AR_COL, limits, fps_col=x_col)
        add_model_and_strategy_legends(ax, plot_df['strategy'].unique())
        fig.tight_layout()

        # Variant names for each point (no overlap on the chart).
        # display(
        #     plot_df[['model', 'variant', 'post_processed', FPS_COL, AR_COL]]
        #     .sort_values([AR_COL, FPS_COL], ascending=[False, False])
        #     .reset_index(drop=True)
        #     .style.format({AR_COL: '{:.3f}', FPS_COL: '{:.1f}'})
        #     .set_caption('Points on plot (hover-free lookup)')
       
        # )
        show_and_save_plot(fig, 'ar_vs_fps')


## AP and AR vs e2e FPS

The AP and AR charts side by side. Each panel keeps its own y-axis range. The model legend is drawn once, to the right of the recall panel.

`COMBINED_AP_AR_EXPORTS` reloads COCO, CrowdPose, OCHuman, EMDB, and PoseTrack21 and reuses the axis bounds from the separate AP and AR figures. `EXCLUDE_MODELS` still applies. Detector filtering does not: those figures include every detector. The EMDB smoothing entry keeps the `smoothing` model group and the raw, EMA, One Euro, and SmoothNet (window 8 and 16) runs. Marker shapes are assigned from the strategies in that export, so the post-processing legend matches the separate smoothing figures.

In [ ]:
_RESULTS = '/local/home/nkoefarago/mmpose/benchmark/results'
# Axis bounds copied from the previous separate AP and AR figures.
COMBINED_AP_AR_EXPORTS = {
    'coco_ap_ar_vs_fps': {
        'files': [
            f'{_RESULTS}/20260610_coco_e2e.json',
            f'{_RESULTS}/20260610_coco_topdown.json',
            f'{_RESULTS}/20260815_coco_e2e.json',
        ],
        'limits': {
            'fps': (0, 227.997632299572),
            'coco/AP': (0.50, 0.82),
            'coco/AR': (0.54, 0.84),
        },
    },
    'crowdpose_ap_ar_vs_fps': {
        'files': [
            f'{_RESULTS}/20260610_crowdpose_e2e.json',
            f'{_RESULTS}/20260610_crowdpose_topdown.json',
            f'{_RESULTS}/20260815_crowdpose_e2e.json',
        ],
        'limits': {
            'fps': (0, 224.08708331628563),
            'coco/AP': (0.50, 0.81),
            'coco/AR': (0.59, 0.88),
        },
    },
    'ochuman_ap_ar_vs_fps': {
        'files': [
            f'{_RESULTS}/20260611_ochuman_e2e.json',
            f'{_RESULTS}/20260611_ochuman_topdown.json',
            f'{_RESULTS}/20261018_ochuman_e2e.json',
        ],
        'limits': {
            'fps': (0, 250.5095069827199),
            'coco/AP': (0.25, 0.47),
            'coco/AR': (0.57, 0.90),
        },
    },
    'emdb_ap_ar_vs_fps': {
        'files': [
            f'{_RESULTS}/20260715_emdb_e2e.json',
            f'{_RESULTS}/20260715_emdb_topdown.json',
            f'{_RESULTS}/20260815_emdb_e2e.json',
            f'{_RESULTS}/20260806_emdb_video.json',
        ],
        'limits': {
            'fps': (0, 259.1562644954748),
            'coco/AP': (0.86, 0.98),
            'coco/AR': (0.87, 1.0),
        },
    },
    'posetrack21_ap_ar_vs_fps': {
        'files': [
            f'{_RESULTS}/20261028_posetrack21_e2e.json',
            f'{_RESULTS}/20261028_posetrack21_topdown.json',
            f'{_RESULTS}/20261029_posetrack21_tracking.json',
            f'{_RESULTS}/20261029_posetrack21_video.json',
        ],
        'limits': {
            'fps': (0, 215.96771896863766),
            'coco/AP': (0.2528756977410557, 0.8224420651987643),
            'coco/AR': (0.270553633813482, 0.8473839640662915),
        },
    },
    'emdb_smoothing_ap_ar_vs_fps': {
        'files': [
            f'{_RESULTS}/20260715_emdb_e2e.json',
            f'{_RESULTS}/20260715_emdb_topdown.json',
            f'{_RESULTS}/20260815_emdb_e2e.json',
            f'{_RESULTS}/20260806_emdb_video.json',
            f'{_RESULTS}/20261026_emdb_postproc_ema.json',
            f'{_RESULTS}/20261026_emdb_postproc_oneeuro.json',
            f'{_RESULTS}/20261026_emdb_postproc_smoothnetw8.json',
            f'{_RESULTS}/20261026_emdb_postproc_smoothnetw16.json',
            f'{_RESULTS}/20261104_emdb_postproc_gpkalman.json'
        ],
        'pairs': MODEL_GROUPS['smoothing'],
        'limits': {
            'fps': (0, 100),
            'coco/AP': (0.52, 0.965),
            'coco/AR': (0.76, 0.98),
        },
    },
}

def _load_combined_export(spec):
    """Load an export spec, then give each of its strategies a distinct marker.

    Marker shapes are fixed from whichever results cell 1 loaded. A smoothing
    export adds strategies that load does not contain, so rebuild the map from
    the strategies actually plotted. ``_build_strategy_markers`` sorts those
    names the same way as the separate smoothing figures.
    """
    global STRATEGY_MARKERS
    records = []
    for path in spec['files']:
        if not os.path.isfile(path):
            raise FileNotFoundError(f'Results file not found: {path}')
        records.extend(load_eval_results(path))
    export_df = exclude_models(pd.DataFrame(records), EXCLUDE_MODELS)
    if spec.get('pairs'):
        export_df = filter_by_model_variants(export_df, spec['pairs'])
    strategies = export_df['strategy'].dropna().unique()
    if len(set(strategies)) > 1:
        STRATEGY_MARKERS = _build_strategy_markers(strategies)
    return export_df

for _name, _spec in COMBINED_AP_AR_EXPORTS.items():
    plot_ap_ar_vs_fps(_load_combined_export(_spec), _spec['limits'], _name)


## MPJVE and MPJAE vs e2e FPS

The bbox-normalised bMPJVE and bMPJAE charts side by side for EMDB and PoseTrack21. Each panel keeps its own y-axis range, using the bounds from the separate figures. The model legend is drawn once, to the right of the bMPJAE panel, with the post-processing legend under it when the export has more than one strategy. The result files are the same ones as the combined AP and AR exports. PoseTrack21 also has a crop, with the tighter bounds from `posetrack21_crop_mpjve_vs_fps` and `posetrack21_crop_mpjae_vs_fps`. The EMDB smoothing entry uses the same model group and strategy markers as `emdb_smoothing_ap_ar_vs_fps`.

In [ ]:
_emdb = COMBINED_AP_AR_EXPORTS['emdb_ap_ar_vs_fps']
_posetrack = COMBINED_AP_AR_EXPORTS['posetrack21_ap_ar_vs_fps']
# Y bounds copied from the previous separate bMPJVE and bMPJAE figures.
# PoseTrack uses the full-range figures (posetrack21_all_*), which share the
# AP/AR x-axis. EMDB y-limits were set by hand on those figures.
COMBINED_MPJ_EXPORTS = {
    'emdb_mpjve_mpjae_vs_fps': {
        'files': _emdb['files'],
        'limits': {
            'fps': _emdb['limits']['fps'],
            'bMPJVE': (0, 0.017),
            'bMPJAE': (0, 0.027),
        },
    },
    'posetrack21_mpjve_mpjae_vs_fps': {
        'files': _posetrack['files'],
        'limits': {
            'fps': _posetrack['limits']['fps'],
            'bMPJVE': (0.007750366816028301, 0.12132131894571564),
            'bMPJAE': (0.010080383331682612, 0.22364483477580357),
        },
    },
    'posetrack21_crop_mpjve_mpjae_vs_fps': {
        'files': _posetrack['files'],
        'limits': {
            'fps': (0, 125),
            'bMPJVE': (0, 0.027),
            'bMPJAE': (0, 0.04),
        },
    },
    'emdb_smoothing_mpjve_mpjae_vs_fps': {
        'files': COMBINED_AP_AR_EXPORTS['emdb_smoothing_ap_ar_vs_fps']['files'],
        'pairs': COMBINED_AP_AR_EXPORTS['emdb_smoothing_ap_ar_vs_fps']['pairs'],
        'limits': {
            'fps': (0, 100),
            'bMPJVE': (0.004, 0.012),
            'bMPJAE': (0.002, 0.018),
        },
    },
}

for _name, _spec in COMBINED_MPJ_EXPORTS.items():
    plot_two_metrics_vs_fps(
        _load_combined_export(_spec), _spec['limits'], _name, 'bMPJVE', 'bMPJAE',
    )


## AP vs e2e FPS (by detector)

Same chart as above, with marker shape encoding the detector instead of the post-processing strategy. Legend text comes from `DETECTOR_DISPLAY_NAMES`.


In [ ]:
AP_COL = 'coco/AP'
FPS_COL = 'perf/e2e/fps'
# Set to a positive int to label only the N highest-AP points (0 = no point labels).
ANNOTATE_TOP_N = 0

missing = [c for c in (AP_COL, FPS_COL) if c not in df_plot.columns]
if missing:
    print(f'Missing columns for plot: {missing}')
    print('Available metric columns:', METRIC_COLS)
else:
    plot_df = latest_plot_runs(df_plot).dropna(subset=[AP_COL, FPS_COL])

    if plot_df.empty:
        print(f'No rows with both {AP_COL} and {FPS_COL}.')
    else:
        models = plot_df['model'].unique()

        fig, ax = plt.subplots(figsize=(8, 5))
        for model in models:
            sub = plot_df[plot_df['model'] == model]
            scatter_runs_by_detector(ax, sub, FPS_COL, AP_COL, model, show_model_label=True)

        if ANNOTATE_TOP_N > 0:
            top = plot_df.nlargest(ANNOTATE_TOP_N, AP_COL)
            for _, row in top.iterrows():
                ax.annotate(
                    latex_text(row['variant']),
                    (row[FPS_COL], row[AP_COL]),
                    xytext=(4, 4),
                    textcoords='offset points',
                    fontsize=7,
                    color=color_for_model(row['model']),
                )

        ax.set_xlabel('FPS')
        ax.set_ylabel('AP')
        ax.set_title('Average precision vs end-to-end throughput')
        ax.grid(True, alpha=0.3)
        apply_plot_axis_limits(ax, FPS_COL, AP_COL, PLOT_AXIS_LIMITS)
        add_model_and_detector_legends(
            ax, plot_df['variant'].map(detector_label_from_run_name),
        )
        fig.tight_layout()
        show_and_save_plot(fig, 'ap_vs_fps_by_detector')


## AR vs inference speed (by detector)

Same chart as above, with marker shape encoding the detector instead of the post-processing strategy. Legend text comes from `DETECTOR_DISPLAY_NAMES`.

Default x-axis is the **base model's** e2e FPS (`perf/e2e/fps`), including for post-processed runs. Set `INCLUDE_POSTPROC_TIME = True` to plot combined model + post-processing throughput instead: `num_frames / (num_frames / e2e_fps + postproc_total_s)`.


In [ ]:
AR_COL = 'coco/AR'
# False: x-axis is the base model's e2e FPS (perf/e2e/fps), including for
# post-processed runs. True: x-axis is combined model + post-processing FPS
# num_frames / (num_frames / e2e_fps + postproc_total_s); raw runs without
# postproc time are unchanged.
INCLUDE_POSTPROC_TIME = False
# Set to a positive int to label only the N highest-AR points (0 = no point labels).
ANNOTATE_TOP_N = 0

if AR_COL not in df_plot.columns:
    print(f'Missing column for plot: {AR_COL}')
    print('Available metric columns:', METRIC_COLS)
else:
    plot_df = add_throughput_fps(
        latest_plot_runs(df_plot),
        include_postproc_time=INCLUDE_POSTPROC_TIME,
    )
    x_col = COMBINED_FPS_COL
    plot_df = plot_df.dropna(subset=[AR_COL, x_col])

    if plot_df.empty:
        print(f'No rows with both {AR_COL} and throughput FPS.')
    else:
        models = plot_df['model'].unique()

        fig, ax = plt.subplots(figsize=(8, 5))
        for model in models:
            sub = plot_df[plot_df['model'] == model]
            scatter_runs_by_detector(ax, sub, x_col, AR_COL, model, show_model_label=True)

        if ANNOTATE_TOP_N > 0:
            top = plot_df.nlargest(ANNOTATE_TOP_N, AR_COL)
            for _, row in top.iterrows():
                ax.annotate(
                    latex_text(row['variant']),
                    (row[x_col], row[AR_COL]),
                    xytext=(4, 4),
                    textcoords='offset points',
                    fontsize=7,
                    color=color_for_model(row['model']),
                )

        if INCLUDE_POSTPROC_TIME:
            xlabel = 'combined FPS (model + post-processing)'
            title_fps = 'combined model + post-processing throughput'
        else:
            xlabel = 'e2e FPS'
            title_fps = 'end-to-end throughput'

        ax.set_xlabel(xlabel)
        ax.set_ylabel('AR')
        ax.set_title(
            f'Recall vs {title_fps}'
        )
        ax.grid(True, alpha=0.3)

        limits = dict(PLOT_AXIS_LIMITS)
        if INCLUDE_POSTPROC_TIME and plot_df[x_col].notna().any():
            vals = plot_df[x_col].dropna()
            limits['fps'] = _axis_limit(
                vals.min(), vals.max(), padding=AXIS_PADDING)
        apply_plot_axis_limits(ax, x_col, AR_COL, limits, fps_col=x_col)
        add_model_and_detector_legends(
            ax, plot_df['variant'].map(detector_label_from_run_name),
        )
        fig.tight_layout()
        show_and_save_plot(fig, 'ar_vs_fps_by_detector')


## EMDB temporal metric vs e2e FPS

In [ ]:
METRIC_COL_NAME = resolve_temporal_metric_col(df_plot, 'bMPJVE')
FPS_COL = 'perf/e2e/fps'
# Set to a positive int to label only the N best (lowest) points (0 = no point labels).
ANNOTATE_TOP_N = 0

if METRIC_COL_NAME is None:
    print('No bMPJVE column found (expected emdb/bMPJVE or temporal/bMPJVE).')
    print('Available metric columns:', METRIC_COLS)
elif FPS_COL not in df_plot.columns:
    print(f'Missing column for plot: {FPS_COL}')
    print('Available metric columns:', METRIC_COLS)
else:
    plot_df = latest_plot_runs(df_plot).dropna(subset=[METRIC_COL_NAME, FPS_COL])

    if plot_df.empty:
        print(f'No rows with both {METRIC_COL_NAME} and {FPS_COL}.')
    else:
        models = plot_df['model'].unique()

        fig, ax = plt.subplots(figsize=(8, 5))
        for model in models:
            sub = plot_df[plot_df['model'] == model]
            scatter_runs(ax, sub, FPS_COL, METRIC_COL_NAME, model, show_model_label=True)

        if ANNOTATE_TOP_N > 0:
            top = plot_df.nsmallest(ANNOTATE_TOP_N, METRIC_COL_NAME)
            for _, row in top.iterrows():
                ax.annotate(
                    latex_text(row['variant']),
                    (row[FPS_COL], row[METRIC_COL_NAME]),
                    xytext=(4, 4),
                    textcoords='offset points',
                    fontsize=7,
                    color=color_for_model(row['model']),
                )

        ax.set_xlabel('FPS')
        ax.set_ylabel(metric_label(METRIC_COL_NAME))
        ax.set_title(
            f'{metric_label(METRIC_COL_NAME)} vs end-to-end throughput'
        )
        ax.grid(True, alpha=0.3)
        apply_plot_axis_limits(ax, FPS_COL, METRIC_COL_NAME, PLOT_AXIS_LIMITS)
        add_model_and_strategy_legends(ax, plot_df['strategy'].unique())
        fig.tight_layout()

        # display(
        #     plot_df[['model', 'variant', 'post_processed', FPS_COL, METRIC_COL_NAME]]
        #     .sort_values([METRIC_COL_NAME, FPS_COL], ascending=[True, False])
        #     .reset_index(drop=True)
        #     .style.format({METRIC_COL_NAME: '{:.4f}', FPS_COL: '{:.1f}'})
        #     .set_caption('Points on plot (hover-free lookup)')
        # )
        show_and_save_plot(fig, 'temporal_vs_fps')


## HOTA vs inference speed

Default x-axis is the **base model's** e2e FPS (`perf/e2e/fps`), including for post-processed runs. Set `INCLUDE_POSTPROC_TIME = True` to plot combined model + post-processing throughput instead: `num_frames / (num_frames / e2e_fps + postproc_total_s)`.

In [ ]:
HOTA_COL = 'tracking/HOTA'
# False: x-axis is the base model's e2e FPS (perf/e2e/fps), including for
# post-processed runs. True: x-axis is combined model + post-processing FPS
# num_frames / (num_frames / e2e_fps + postproc_total_s); raw runs without
# postproc time are unchanged.
INCLUDE_POSTPROC_TIME = True
# Set to a positive int to label only the N highest-HOTA points (0 = no point labels).
ANNOTATE_TOP_N = 0

if HOTA_COL not in df_plot.columns:
    print(f'Missing column for plot: {HOTA_COL}')
    print('Available metric columns:', METRIC_COLS)
else:
    plot_df = add_throughput_fps(
        latest_plot_runs(df_plot),
        include_postproc_time=INCLUDE_POSTPROC_TIME,
    )
    x_col = COMBINED_FPS_COL
    plot_df = plot_df.dropna(subset=[HOTA_COL, x_col])

    if plot_df.empty:
        print(f'No rows with both {HOTA_COL} and throughput FPS.')
    else:
        models = plot_df['model'].unique()

        fig, ax = plt.subplots(figsize=(8, 5))
        for model in models:
            sub = plot_df[plot_df['model'] == model]
            scatter_runs(ax, sub, x_col, HOTA_COL, model, show_model_label=True)

        if ANNOTATE_TOP_N > 0:
            top = plot_df.nlargest(ANNOTATE_TOP_N, HOTA_COL)
            for _, row in top.iterrows():
                ax.annotate(
                    latex_text(row['variant']),
                    (row[x_col], row[HOTA_COL]),
                    xytext=(4, 4),
                    textcoords='offset points',
                    fontsize=7,
                    color=color_for_model(row['model']),
                )

        if INCLUDE_POSTPROC_TIME:
            xlabel = 'combined FPS (model + post-processing)'
            title_fps = 'combined model + post-processing throughput'
        else:
            xlabel = 'base-model e2e FPS'
            title_fps = 'base-model throughput'

        ax.set_xlabel(xlabel)
        ax.set_ylabel('HOTA')
        ax.set_title(
            f'HOTA vs {title_fps} (latest run per model / variant)'
        )
        ax.grid(True, alpha=0.3)

        limits = dict(PLOT_AXIS_LIMITS)
        if INCLUDE_POSTPROC_TIME and plot_df[x_col].notna().any():
            vals = plot_df[x_col].dropna()
            limits['fps'] = _axis_limit(
                vals.min(), vals.max(), padding=AXIS_PADDING)
        apply_plot_axis_limits(ax, x_col, HOTA_COL, limits, fps_col=x_col)

        add_model_and_strategy_legends(ax, plot_df['strategy'].unique())
        fig.tight_layout()
        show_and_save_plot(fig, 'hota_vs_fps')


## HOTA vs FPS by model combination

Each color is one model combination: the pose model plus its tracker or post-processing method. Size variants of that combination share the color. DETRPose n/s/m/l/x with ByteTrack is one color; DETRPose with BoT-SORT is another. Capacity tokens count as size (`n`/`s`/`m`/`l`/`x`, nano–huge, ResNet width, ViT size, HRNet width). The detector family stays in the combination, so RF-DETR and RTMDet are different colors. AlphaPose and OpenPifPaf are each one color across backbone and tracker variants.

Runs that share a variant name and differ only by `post_config` stay separate (PCT huge + RF-DETR with several trackers). The x-axis is the reported e2e FPS (`perf/e2e/fps`). These runs timed the pose model and the tracker together, so that number is the whole pipeline. GP-Kalman SORT and DETRPose-MOTIP are drawn as stars when they are in the data.

Re-run the load cell once so `post_config` is on the dataframe.


In [ ]:
import matplotlib.colors as mcolors

# Capacity tokens dropped so size variants share a color. Detector family,
# backbone family, training tags, and the tracker / post-processing method stay.
_COMBINATION_SIZE_TOKENS = {
    'n', 's', 'm', 'l', 'x', 't',
    'nano', 'tiny', 'small', 'medium', 'large', 'huge', 'xlarge', '2xlarge',
    'base',
    'w16', 'w32', 'w48',
    'res18', 'res50', 'res101', 'res152',
    'r18', 'r50', 'r101',
    'k16', 'k30',
    'vits', 'vitb', 'vitl', 'vith',
}
_COMBINATION_SIZE_RE = re.compile(r'^(?:0\.\d+|\d+)b$')
_COMBINATION_IGNORE_TOKENS = {'published'}
# Every run of these models is one combination. Backbone, detector, and
# built-in tracker names are size/architecture variants, not new methods.
_SINGLE_COLOR_MODELS = {'AlphaPose', 'OpenPifPaf'}
_STACK_TOKEN_DISPLAY = {
    'rfdetr': 'RF-DETR',
    'rtmdet': 'RTMDet',
    'yolo26': 'YOLO26',
    'fasterrcnn': 'Faster R-CNN',
    'motip': 'MOTIP',
    'dancetrack': 'DanceTrack',
    'posetrack21': 'PoseTrack21',
    'posetrack17': 'PoseTrack17',
    'coco': 'COCO',
    'fastpose': 'FastPose',
    'duc': 'DUC',
    'dcn': 'DCN',
    'hrnet': 'HRNet',
    'hgnetv2': 'HGNetv2',
    'trackingpose': 'tracking-pose',
    'posesimilarity': 'pose-similarity',
    'shufflenetv2': 'ShuffleNetv2',
    'tshufflenetv2': 'T-ShuffleNetv2',
    'swin': 'Swin',
    'cspnext': 'CSPNeXt',
    'resnet': 'ResNet',
    '2xresnet': '2xResNet',
    '4xresnet': '4xResNet',
    'mobilenetv2': 'MobileNetv2',
    'vipnas': 'ViPNAS',
}
_POSTPROC_DISPLAY = {
    'bytetrack': 'ByteTrack',
    'deepocsort': 'Deep OC-SORT',
    'botsort': 'BoT-SORT',
    'ocsort': 'OC-SORT',
    'occluboost': 'OccluBoost',
    'gp-kalman-sort': 'GP-Kalman SORT',
    'gp-kalman': 'GP-Kalman',
    'pgpt': 'PGPT',
    'pgpt-geom': 'PGPT geom',
    'pgpt-tuned': 'PGPT tuned',
    'lighttrack-l2': 'LightTrack L2',
    'lighttrack-sgcn': 'LightTrack SGCN',
    'detect-and-track-iou': 'Detect-and-Track IoU',
    'detect-and-track-iou-tuned': 'Detect-and-Track IoU tuned',
    'detect-and-track-greedy-pck': 'Detect-and-Track greedy PCK',
    'detect-and-track-cnn': 'Detect-and-Track CNN',
    'oks-nms-pgpt': 'OKS-NMS PGPT',
    'oks-nms-pgpt-geom': 'OKS-NMS PGPT geom',
    'oks-nms-boxmot-occluboost': 'OKS-NMS OccluBoost',
    'oks-nms': 'OKS-NMS',
}
_VARIANT_SIZE_RANK = {
    'n': 0, 's': 1, 'm': 2, 'l': 3, 'x': 4, 't': 5,
    'nano': 6, 'tiny': 7, 'small': 8, 'medium': 9, 'large': 10,
    'huge': 11, 'xlarge': 12, '2xlarge': 13, 'base': 14,
}


def _normalize_postproc_name(post_config) -> str:
    """Stem of a post-processing config, with the boxmot_ prefix removed."""
    if post_config is None:
        return ''
    try:
        if pd.isna(post_config):
            return ''
    except (TypeError, ValueError):
        pass
    text = str(post_config).strip()
    if not text or text.lower() in {'none', 'nan'}:
        return ''
    stem = os.path.splitext(os.path.basename(text))[0].replace('_', '-')
    if stem.startswith('boxmot-'):
        stem = stem[len('boxmot-'):]
    if stem in {'', 'empty', 'none'}:
        return ''
    return stem


def _normalize_capacity_token(token):
    """Drop a pure size token, or strip a glued width (hrnetw48 -> hrnet)."""
    if token in _COMBINATION_SIZE_TOKENS or _COMBINATION_SIZE_RE.match(token):
        return None
    match = re.fullmatch(r'(hrnet)w\d+', token)
    if match:
        return match.group(1)
    match = re.fullmatch(r'((?:t)?shufflenetv2)k\d+', token)
    if match:
        return match.group(1)
    match = re.fullmatch(r'(\d+xresnet)\d+', token)
    if match:
        return match.group(1)
    match = re.fullmatch(r'(resnet)\d+', token)
    if match:
        return match.group(1)
    return token


def _remove_token_sequence(tokens, sequence):
    if not sequence:
        return tokens
    width = len(sequence)
    for i in range(len(tokens) - width + 1):
        if tokens[i:i + width] == sequence:
            return tokens[:i] + tokens[i + width:]
    return tokens


def model_combination_key(model, variant, post_config='') -> tuple:
    """(model, size-stripped stack, post-processing) identity.

    DETRPose n/s/m/l/x + ByteTrack share one key. ByteTrack and BoT-SORT do
    not. A detector size is stripped; the detector family is kept.
    AlphaPose and OpenPifPaf collapse to the model name alone.
    """
    if str(model) in _SINGLE_COLOR_MODELS:
        return (str(model), '', '')
    postproc = _normalize_postproc_name(post_config)
    if variant is None:
        raw = ''
    else:
        try:
            raw = '' if pd.isna(variant) else str(variant)
        except (TypeError, ValueError):
            raw = str(variant)
    tokens = [tok for tok in raw.lower().replace('_', '-').split('-') if tok]
    if postproc:
        tokens = _remove_token_sequence(tokens, postproc.split('-'))
    kept = []
    for token in tokens:
        normalized = _normalize_capacity_token(token)
        if normalized is None or normalized in _COMBINATION_IGNORE_TOKENS:
            continue
        kept.append(normalized)
    return (str(model), '-'.join(kept), postproc)


def combination_display_name(key) -> str:
    """Legend label for a model_combination_key."""
    model, stack, postproc = key
    parts = [str(model)]
    if stack:
        parts.append(' '.join(
            _STACK_TOKEN_DISPLAY.get(token, token) for token in stack.split('-')
        ))
    if postproc:
        parts.append(_POSTPROC_DISPLAY.get(postproc, postproc))
    return ' + '.join(parts)


# Qualitative colors ordered so the first entries are far apart in hue.
# Later entries are lighter or darker variants for plots with many combinations.
_COMBINATION_PALETTE = [
    '#E41A1C',  # red
    '#377EB8',  # blue
    '#4DAF4A',  # green
    '#FF7F00',  # orange
    '#984EA3',  # purple
    '#00BFC4',  # cyan
    '#F781BF',  # pink
    '#A65628',  # brown
    '#FFD92F',  # yellow
    '#1B4F72',  # navy
    '#E7298A',  # magenta
    '#006837',  # dark green
    '#222222',  # black
    '#FB9A99',  # salmon
    '#B2DF8A',  # light green
    '#FDBF6F',  # apricot
    '#CAB2D6',  # lavender
    '#B15928',  # rust
    '#666666',  # gray
    '#B3DE69',  # lime
]


def combination_colors(labels):
    """One high-contrast color per combination, in legend order."""
    ordered = sorted(set(labels))
    palette = [mcolors.to_rgb(hex_color) for hex_color in _COMBINATION_PALETTE]
    return {
        label: palette[i % len(palette)]
        for i, label in enumerate(ordered)
    }


def latest_runs_by_postproc(data):
    """Latest run per (model, variant, post_config).

    Several trackers can share one variant name and differ only by post_config
    (PCT huge-rfdetr). Grouping on variant alone would keep a single tracker.
    """
    frame = data.copy()
    if 'post_config' not in frame.columns:
        frame['post_config'] = ''
    frame['post_config'] = frame['post_config'].fillna('').astype(str)
    return (
        frame.sort_values('timestamp')
        .groupby(['model', 'variant', 'post_config'], sort=False)
        .last()
        .reset_index()
    )


def _variant_size_sort_key(variant):
    first = str(variant).split('-')[0].lower()
    return (_VARIANT_SIZE_RANK.get(first, 50), str(variant))


HOTA_COL = 'tracking/HOTA'
# Reported e2e FPS. These runs timed the pose model and the tracker together,
# so perf/e2e/fps is the whole pipeline, not a base-model time to add postproc onto.
FPS_COL = 'perf/e2e/fps'
# Set to a positive int to label only the N highest-HOTA points (0 = no labels).
ANNOTATE_TOP_N = 0
# Drawn as stars. DETRPose-MOTIP is the model name; GP-Kalman SORT is the tracker.
_EMPHASIS_MODELS = {'DETRPose-MOTIP'}
_EMPHASIS_POSTPROC = {'gp-kalman-sort'}


def _emphasis_marker(sub) -> str:
    """Star for GP-Kalman SORT and DETRPose-MOTIP, circle otherwise."""
    row = sub.iloc[0]
    if str(row['model']) in _EMPHASIS_MODELS:
        return '*'
    if canonical_postproc_name(row['post_config']) in _EMPHASIS_POSTPROC:
        return '*'
    return 'o'

if 'post_config' not in df_plot.columns:
    print('post_config is missing. Re-run the load cell, then this one.')
elif HOTA_COL not in df_plot.columns or FPS_COL not in df_plot.columns:
    missing = [c for c in (HOTA_COL, FPS_COL) if c not in df_plot.columns]
    print(f'Missing columns for plot: {missing}')
    print('Available metric columns:', METRIC_COLS)
else:
    plot_df = latest_runs_by_postproc(df_plot)
    plot_df = plot_df.dropna(subset=[HOTA_COL, FPS_COL]).copy()
    plot_df['combination'] = [
        combination_display_name(model_combination_key(model, variant, post))
        for model, variant, post in zip(
            plot_df['model'], plot_df['variant'], plot_df['post_config'],
        )
    ]

    if plot_df.empty:
        print(f'No rows with both {HOTA_COL} and {FPS_COL}.')
    else:
        print(f'{plot_df["combination"].nunique()} model combination(s):')
        for label, group in plot_df.groupby('combination', sort=True):
            variants = sorted(group['variant'].unique(), key=_variant_size_sort_key)
            print(f'  {label}: {", ".join(variants)}')

        colors = combination_colors(plot_df['combination'])
        n_labels = len(colors)
        fig, ax = plt.subplots(figsize=(12, 6))
        for label in sorted(colors):
            sub = plot_df[plot_df['combination'] == label]
            marker = _emphasis_marker(sub)
            ax.scatter(
                sub[FPS_COL],
                sub[HOTA_COL],
                s=140 if marker == '*' else 32,
                alpha=0.92,
                color=colors[label],
                edgecolors='white',
                linewidths=0.4,
                marker=marker,
                zorder=3 if marker == '*' else 2,
                label=latex_text(label),
            )

        if ANNOTATE_TOP_N > 0:
            top = plot_df.nlargest(ANNOTATE_TOP_N, HOTA_COL)
            for _, row in top.iterrows():
                ax.annotate(
                    latex_text(row['variant']),
                    (row[FPS_COL], row[HOTA_COL]),
                    xytext=(4, 4),
                    textcoords='offset points',
                    fontsize=7,
                    color=colors[row['combination']],
                )

        ax.set_xlabel('FPS')
        ax.set_ylabel('HOTA')
        ax.set_title(
            'HOTA vs end-to-end throughput (one color per model combination)'
        )
        ax.grid(True, alpha=0.3)
        apply_plot_axis_limits(ax, FPS_COL, HOTA_COL, PLOT_AXIS_LIMITS)
        ax.legend(
            title='combination',
            bbox_to_anchor=(1.02, 1),
            loc='upper left',
            fontsize=7,
            title_fontsize=8,
            framealpha=0.9,
            labelspacing=0.25,
            borderpad=0.4,
        )
        fig.tight_layout()
        show_and_save_plot(
            fig, 'hota_vs_fps_by_combination', bbox_inches='tight')


## ID switches per 100 frames vs e2e FPS

In [ ]:
IDSW_COL = 'tracking/IDSwitchesPer100Frames'
FPS_COL = 'perf/e2e/fps'
# Set to a positive int to label only the N best (fewest ID switches) points (0 = no point labels).
ANNOTATE_TOP_N = 0

missing = [c for c in (IDSW_COL, FPS_COL) if c not in df_plot.columns]
if missing:
    print(f'Missing columns for plot: {missing}')
    print('Available metric columns:', METRIC_COLS)
else:
    plot_df = latest_plot_runs(df_plot).dropna(subset=[IDSW_COL, FPS_COL])

    if plot_df.empty:
        print(f'No rows with both {IDSW_COL} and {FPS_COL}.')
    else:
        models = plot_df['model'].unique()

        fig, ax = plt.subplots(figsize=(8, 5))
        for model in models:
            sub = plot_df[plot_df['model'] == model]
            scatter_runs(ax, sub, FPS_COL, IDSW_COL, model, show_model_label=True)

        if ANNOTATE_TOP_N > 0:
            top = plot_df.nsmallest(ANNOTATE_TOP_N, IDSW_COL)
            for _, row in top.iterrows():
                ax.annotate(
                    latex_text(row['variant']),
                    (row[FPS_COL], row[IDSW_COL]),
                    xytext=(4, 4),
                    textcoords='offset points',
                    fontsize=7,
                    color=color_for_model(row['model']),
                )

        ax.set_xlabel('e2e FPS')
        ax.set_ylabel('ID switches / 100 frames')
        ax.set_title(
            'ID switches per 100 frames vs end-to-end throughput '
            '(latest run per model / variant)'
        )
        ax.grid(True, alpha=0.3)
        apply_plot_axis_limits(ax, FPS_COL, IDSW_COL, PLOT_AXIS_LIMITS)
        add_model_and_strategy_legends(ax, plot_df['strategy'].unique())
        fig.tight_layout()

        # display(
        #     plot_df[['model', 'variant', 'post_processed', FPS_COL, IDSW_COL]]
        #     .sort_values([IDSW_COL, FPS_COL], ascending=[True, False])
        #     .reset_index(drop=True)
        #     .style.format({IDSW_COL: '{:.4f}', FPS_COL: '{:.1f}'})
        #     .set_caption('Points on plot (hover-free lookup)')
        # )
        show_and_save_plot(fig, 'idsw_vs_fps')


## EMDB keypoint trajectory over time

Loads a benchmark prediction export (`manifest.json` + `frames.json` from `benchmark_e2e.py`) and plots one keypoint's **x** and **y** image coordinates across frames in a sequence.

In [ ]:
import json
import os.path as osp

# ── Configuration ────────────────────────────────────────────────────────────
# Directory with manifest.json and frames.json (benchmark_e2e export).
PRED_DIR = (
    '/local/home/nkoefarago/mmpose/'
    'benchmark/predictions/20260622_emdb_topdown/ViTPose-small-rfdetr__postproc'
)
SEQUENCE = None  # None = first sequence in the bundle, or e.g. 'P1/14_outdoor_climb'
KEYPOINT = 'right_wrist'  # name from dataset_meta, or int index (0–16 for COCO-17)
SHOW_GT = True  # overlay matched ground-truth keypoint
MIN_SCORE = 0.0  # skip pred points below this keypoint score


def _load_prediction_bundle(pred_dir: str):
    manifest_path = osp.join(pred_dir, 'manifest.json')
    frames_path = osp.join(pred_dir, 'frames.json')
    if not osp.isfile(manifest_path) or not osp.isfile(frames_path):
        raise FileNotFoundError(
            f'Expected manifest.json and frames.json in {pred_dir}')
    with open(manifest_path, encoding='utf-8') as f:
        manifest = json.load(f)
    with open(frames_path, encoding='utf-8') as f:
        frames = json.load(f)
    return manifest, frames


def _emdb_sequence_name(img_path: str) -> str:
    """P1/14_outdoor_climb/images/00042.jpg -> P1/14_outdoor_climb."""
    parts = img_path.replace('\\', '/').split('/')
    if len(parts) >= 3 and parts[-2] == 'images':
        return '/'.join(parts[:-2])
    return osp.dirname(img_path)


def _emdb_frame_index(img_path: str) -> int:
    basename = osp.splitext(osp.basename(img_path))[0]
    if basename.startswith('image_'):
        return int(basename.split('_')[-1])
    return int(basename)


def _keypoint_index(name_or_idx, dataset_meta: dict) -> int:
    if isinstance(name_or_idx, int):
        return name_or_idx
    id2name = dataset_meta.get('keypoint_id2name', {})
    for idx, name in id2name.items():
        if name == name_or_idx:
            return int(idx)
    for idx, info in dataset_meta.get('keypoint_info', {}).items():
        if info.get('name') == name_or_idx:
            return int(idx)
    names = list(id2name.values()) or [
        info.get('name') for info in dataset_meta.get('keypoint_info', {}).values()
    ]
    raise ValueError(
        f'Keypoint {name_or_idx!r} not found. Available: {names}')


def _pick_instance_idx(instances, matches, role: str):
    idx_key = 'pred_idx' if role == 'pred' else 'gt_idx'
    if matches:
        return matches[0][idx_key]
    if instances:
        return 0
    return None


def _extract_keypoint_trajectory(
    frames,
    sequence: str,
    kpt_idx: int,
    show_gt: bool,
    min_score: float,
):
    rows = []
    for frame in frames:
        if _emdb_sequence_name(frame['img_path']) != sequence:
            continue

        matches = frame['metrics'].get('matches', [])
        pred_instances = frame['predictions']['instances']
        gt_instances = frame['ground_truth']['instances']

        pred_i = _pick_instance_idx(pred_instances, matches, 'pred')
        if pred_i is None:
            continue

        pred_kpt = pred_instances[pred_i]['keypoints'][kpt_idx]
        pred_score = pred_instances[pred_i].get('keypoint_scores', [1.0])[kpt_idx]
        if pred_score < min_score:
            continue

        ori_shape = frame.get('ori_shape', [0, 0])
        img_h, img_w = int(ori_shape[0]), int(ori_shape[1])

        gt_kpt = None
        if show_gt:
            gt_i = _pick_instance_idx(gt_instances, matches, 'gt')
            if gt_i is not None:
                gt_kpt = gt_instances[gt_i]['keypoints'][kpt_idx]

        rows.append({
            'frame': _emdb_frame_index(frame['img_path']),
            'pred_x': pred_kpt[0],
            'pred_y': pred_kpt[1],
            'pred_score': pred_score,
            'gt_x': gt_kpt[0] if gt_kpt is not None else None,
            'gt_y': gt_kpt[1] if gt_kpt is not None else None,
            'img_h': img_h,
            'img_w': img_w,
        })

    return pd.DataFrame(rows).sort_values('frame').reset_index(drop=True)


pred_dir = osp.abspath(PRED_DIR)
manifest, frames = _load_prediction_bundle(pred_dir)
dataset_meta = manifest.get('dataset_meta', {})
kpt_idx = _keypoint_index(KEYPOINT, dataset_meta)
kpt_name = (
    KEYPOINT if isinstance(KEYPOINT, str)
    else dataset_meta.get('keypoint_id2name', {}).get(str(KEYPOINT), str(KEYPOINT))
)

sequences = sorted({_emdb_sequence_name(f['img_path']) for f in frames})
if not sequences:
    raise ValueError(f'No frames found in {pred_dir}')

sequence = SEQUENCE or sequences[0]
if sequence not in sequences:
    raise ValueError(
        f'Sequence {sequence!r} not in bundle. Available ({len(sequences)}): '
        f'{sequences[:10]}{"..." if len(sequences) > 10 else ""}'
    )

traj = _extract_keypoint_trajectory(
    frames, sequence, kpt_idx, SHOW_GT, MIN_SCORE)

if traj.empty:
    print(f'No trajectory points for sequence {sequence!r} '
          f'(keypoint={kpt_name}, min_score={MIN_SCORE}).')
else:
    model_label = manifest.get('model_name', 'model')
    if manifest.get('model_variant'):
        model_label = f'{model_label}-{manifest["model_variant"]}'

    fig, axes = plt.subplots(3, 1, figsize=(14, 9), sharex=True)
    axes[0].plot(traj['frame'], traj['pred_x'], color='C3', label='prediction')
    if SHOW_GT and traj['gt_x'].notna().any():
        axes[0].plot(
            traj['frame'], traj['gt_x'], color='C0', linestyle='--', label='ground truth')
    axes[0].set_ylabel('x (px)')
    axes[0].set_title(latex_text(f'{kpt_name} — horizontal position'))
    axes[0].grid(True, alpha=0.3)
    axes[0].legend(loc='upper right')

    axes[1].plot(traj['frame'], traj['pred_y'], color='C3', label='prediction')
    if SHOW_GT and traj['gt_y'].notna().any():
        axes[1].plot(
            traj['frame'], traj['gt_y'], color='C0', linestyle='--', label='ground truth')
    axes[1].set_ylabel('y (px)')
    axes[1].set_title(latex_text(f'{kpt_name} — vertical position'))
    axes[1].grid(True, alpha=0.3)
    axes[1].legend(loc='upper right')

    axes[2].plot(traj['frame'], traj['pred_score'], color='C2', label='confidence')
    axes[2].set_xlabel('frame index')
    axes[2].set_ylabel('score')
    axes[2].set_title(latex_text(f'{kpt_name} — keypoint confidence'))
    axes[2].set_ylim(0, 1)
    axes[2].grid(True, alpha=0.3)
    axes[2].legend(loc='upper right')

    fig.suptitle(
        latex_text(
            f'{model_label} | {manifest.get("test_dataset", "emdb")} | {sequence} '
            f'({len(traj)} frames)'
        ),
        y=1.02,
    )
    fig.tight_layout()
    show_and_save_plot(
        fig, f'trajectory_{sequence}_{kpt_name}'.replace('/', '_')
    )

    display(
        traj.head(10).style.format({
            'pred_x': '{:.1f}',
            'pred_y': '{:.1f}',
            'pred_score': '{:.3f}',
            'gt_x': '{:.1f}',
            'gt_y': '{:.1f}',
        }).set_caption(f'First frames — {sequence}')
    )
    if len(sequences) > 1:
        print(f'Other sequences ({len(sequences)} total): {sequences}')


## Score vs. keypoint error (calibration)

Loads a benchmark prediction bundle (`manifest.json` + `frames.json`) and plots the **Euclidean distance between each predicted keypoint and its matched GT keypoint** against the **predicted keypoint score**.

A well-calibrated model should show monotonically *decreasing* error as the score increases (high score → high confidence → small error). The Spearman ρ per keypoint quantifies this: ρ < 0 is good.

In [ ]:
import json
import os.path as osp

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

# ── Configuration ─────────────────────────────────────────────────────────────
# Directory produced by benchmark_e2e.py (must contain manifest.json + frames.json).
CALIB_PRED_DIR = (
    '/local/home/nkoefarago/mmpose/'
    'benchmark/predictions/20260715_emdb_e2e/YOLO26-Pose-large'
)
# Keypoints to include: list of names (str), list of indices (int), or None / 'all'.
CALIB_KEYPOINTS = None
# Normalize pixel distances by the geometric mean of image height × width.
NORMALIZE_BY_IMAGE_SIZE = False
# Number of score buckets for the reliability / calibration curve.
N_BINS = 20
# Bins with fewer samples than this threshold are hidden from the calibration curve.
MIN_SAMPLES_PER_BIN = 10
# Upper percentile used for the error-envelope shading on the calibration curve.
ERROR_PERCENTILE = 75
# Ignore keypoint predictions below this score (skip near-background detections).
MIN_SCORE = 0.0
# Distance values above this percentile are clipped in the density plot only.
MAX_DIST_PERCENTILE = 99


# ── Helpers ───────────────────────────────────────────────────────────────────
def _calib_load_bundle(pred_dir: str):
    for fname in ('manifest.json', 'frames.json'):
        p = osp.join(pred_dir, fname)
        if not osp.isfile(p):
            raise FileNotFoundError(f'Missing {fname} in {pred_dir}')
    with open(osp.join(pred_dir, 'manifest.json')) as f:
        manifest = json.load(f)
    with open(osp.join(pred_dir, 'frames.json')) as f:
        frames = json.load(f)
    return manifest, frames


def _calib_kpt_indices(kpts_cfg, dataset_meta: dict):
    """Return a list of keypoint indices, or None (= all keypoints)."""
    if kpts_cfg is None or kpts_cfg == 'all':
        return None
    id2name = dataset_meta.get('keypoint_id2name', {})
    name2id = {v: int(k) for k, v in id2name.items()}
    result = []
    for kpt in ([kpts_cfg] if not isinstance(kpts_cfg, (list, tuple)) else kpts_cfg):
        if isinstance(kpt, int):
            result.append(kpt)
        elif kpt in name2id:
            result.append(name2id[kpt])
        else:
            raise ValueError(f'Keypoint {kpt!r} not found. Available: {sorted(name2id)}')
    return result


def _calib_collect(frames, kpt_indices, normalize: bool, min_score: float):
    """Iterate over all matched (pred, GT) pairs and collect (score, distance, kpt_id)."""
    scores_out, dists_out, kpt_ids_out = [], [], []

    for frame in frames:
        matches = frame['metrics'].get('matches', [])
        pred_insts = frame['predictions']['instances']
        gt_insts   = frame['ground_truth']['instances']
        ori_shape  = frame.get('ori_shape', [1, 1])
        img_h, img_w = float(ori_shape[0]), float(ori_shape[1])
        scale = (img_h * img_w) ** 0.5 if normalize else 1.0

        for match in matches:
            pred_inst = pred_insts[match['pred_idx']]
            gt_inst   = gt_insts[match['gt_idx']]

            pred_kpts   = pred_inst['keypoints']
            pred_scores = pred_inst.get('keypoint_scores', [1.0] * len(pred_kpts))
            gt_kpts     = gt_inst['keypoints']
            gt_vis      = gt_inst.get('keypoints_visible', [1.0] * len(gt_kpts))

            n_kpts  = min(len(pred_kpts), len(gt_kpts))
            indices = kpt_indices if kpt_indices is not None else range(n_kpts)

            for k in indices:
                if k >= n_kpts:
                    continue
                vis = float(gt_vis[k]) if k < len(gt_vis) else 1.0
                if vis == 0:
                    continue
                score = float(pred_scores[k]) if k < len(pred_scores) else 1.0
                if score < min_score:
                    continue
                dx = float(pred_kpts[k][0]) - float(gt_kpts[k][0])
                dy = float(pred_kpts[k][1]) - float(gt_kpts[k][1])
                dist = (dx * dx + dy * dy) ** 0.5 / scale
                scores_out.append(score)
                dists_out.append(dist)
                kpt_ids_out.append(int(k))

    return (
        np.array(scores_out, dtype=np.float32),
        np.array(dists_out,  dtype=np.float32),
        np.array(kpt_ids_out, dtype=np.int32),
    )


# ── Load & collect ─────────────────────────────────────────────────────────────
_calib_dir  = osp.abspath(CALIB_PRED_DIR)
_manifest_c, _frames_c = _calib_load_bundle(_calib_dir)
_dmeta_c    = _manifest_c.get('dataset_meta', {})
_kpt_idx_c  = _calib_kpt_indices(CALIB_KEYPOINTS, _dmeta_c)

scores, dists, kpt_ids = _calib_collect(_frames_c, _kpt_idx_c, NORMALIZE_BY_IMAGE_SIZE, MIN_SCORE)

if len(scores) == 0:
    raise ValueError(
        f'No (score, distance) pairs collected from {_calib_dir}. '
        'Check matches / visibility / MIN_SCORE.'
    )

dist_clip   = np.percentile(dists, MAX_DIST_PERCENTILE)
dists_plot  = np.clip(dists, 0.0, dist_clip)
dist_unit   = 'normalized distance' if NORMALIZE_BY_IMAGE_SIZE else 'distance (px)'

# Score axis: use the observed range (some models emit unnormalized scores, e.g. PCT).
_score_lo = float(max(MIN_SCORE, scores.min()))
_score_hi = float(scores.max())
if _score_hi <= _score_lo:
    _score_hi = _score_lo + 1e-3
_score_pad = 0.02 * (_score_hi - _score_lo)
_score_xlim = (_score_lo - _score_pad, _score_hi + _score_pad)

_model_label_c = _manifest_c.get('model_name', 'model')
if _manifest_c.get('model_variant'):
    _model_label_c = f'{_model_label_c}-{_manifest_c["model_variant"]}'

print(f'Collected {len(scores):,} (score, distance) pairs from {len(_frames_c)} frames.')
print(f'Score range  : [{scores.min():.3f}, {scores.max():.3f}]')
print(f'Distance range: [{dists.min():.4f}, {dists.max():.4f}]  ({dist_unit})')
if scores.min() < 0 or scores.max() > 1.01:
    print(
        'Note: keypoint scores are outside [0, 1]; '
        'binning and x-limits use the observed score range.'
    )

# ── Binned calibration statistics ──────────────────────────────────────────────
_bin_edges   = np.linspace(_score_lo, _score_hi, N_BINS + 1)
_bin_centers = 0.5 * (_bin_edges[:-1] + _bin_edges[1:])
_bin_mean  = np.full(N_BINS, np.nan)
_bin_p25   = np.full(N_BINS, np.nan)
_bin_p_hi  = np.full(N_BINS, np.nan)
_bin_n     = np.zeros(N_BINS, dtype=int)

for _i, (_lo, _hi) in enumerate(zip(_bin_edges[:-1], _bin_edges[1:])):
    _mask_b = (scores >= _lo) & (scores <= _hi if _i == N_BINS - 1 else scores < _hi)
    _n = int(_mask_b.sum())
    _bin_n[_i] = _n
    if _n >= MIN_SAMPLES_PER_BIN:
        _d = dists[_mask_b]
        _bin_mean[_i] = _d.mean()
        _bin_p25[_i]  = np.percentile(_d, 25)
        _bin_p_hi[_i] = np.percentile(_d, ERROR_PERCENTILE)

_valid = ~np.isnan(_bin_mean)

# ── Plot ───────────────────────────────────────────────────────────────────────
fig, (ax_hex, ax_cal) = plt.subplots(1, 2, figsize=(16, 6))

# Left – hexbin density --------------------------------------------------------
hb = ax_hex.hexbin(scores, dists_plot, gridsize=60, cmap='YlOrRd',
                    mincnt=1, bins='log')
fig.colorbar(hb, ax=ax_hex, label=latex_text(r'log$_{10}$(count)'))

if _valid.any():
    ax_hex.plot(_bin_centers[_valid], _bin_mean[_valid],
                color='C0', linewidth=2, label='mean error per bin', zorder=5)
    ax_hex.fill_between(_bin_centers[_valid], _bin_p25[_valid], _bin_p_hi[_valid],
                         color='C0', alpha=0.25,
                         label=latex_text(f'25th–{ERROR_PERCENTILE}th pct.'))

ax_hex.set_xlabel('predicted score')
ax_hex.set_ylabel(latex_text(
    f'{dist_unit}  (clipped at {MAX_DIST_PERCENTILE}th pct.)'))
ax_hex.set_title(latex_text(
    f'Score vs. keypoint error — {_model_label_c}\n({len(scores):,} keypoints)'))
ax_hex.grid(True, alpha=0.3)
_hex_handles, _hex_labels = ax_hex.get_legend_handles_labels()
if _hex_handles:
    ax_hex.legend(fontsize=9)
ax_hex.set_xlim(*_score_xlim)
ax_hex.set_ylim(0, dist_clip * 1.05)

# Right – calibration curve + sample counts -----------------------------------
ax_twin = ax_cal.twinx()

ax_twin.bar(_bin_centers, _bin_n,
            width=(_bin_edges[1] - _bin_edges[0]) * 0.9,
            color='C7', alpha=0.3, label='sample count')
ax_twin.set_ylabel(latex_text('# keypoints in bin'), color='C7')
ax_twin.tick_params(axis='y', labelcolor='C7')

if _valid.any():
    ax_cal.plot(_bin_centers[_valid], _bin_mean[_valid],
                'o-', color='C1', linewidth=2, markersize=5, label='mean error')
    ax_cal.fill_between(_bin_centers[_valid], _bin_p25[_valid], _bin_p_hi[_valid],
                         color='C1', alpha=0.25,
                         label=latex_text(f'25th–{ERROR_PERCENTILE}th pct.'))

ax_cal.set_xlabel('predicted score')
ax_cal.set_ylabel(dist_unit)
ax_cal.set_title(latex_text(f'Calibration curve — {_model_label_c}'))
ax_cal.grid(True, alpha=0.3)
ax_cal.set_xlim(*_score_xlim)

_lines, _labels   = ax_cal.get_legend_handles_labels()
_bars,  _blabels  = ax_twin.get_legend_handles_labels()
ax_cal.legend(_lines + _bars, _labels + _blabels, fontsize=9, loc='upper right')

fig.tight_layout()
show_and_save_plot(fig, 'score_vs_keypoint_error')

# ── Per-keypoint summary table ─────────────────────────────────────────────────
_id2name_c = _dmeta_c.get('keypoint_id2name', {})
_kpt_rows  = []
for _k in np.unique(kpt_ids):
    _m = kpt_ids == _k
    _s, _d = scores[_m], dists[_m]
    _rho, _pval = stats.spearmanr(_s, _d) if len(_s) > 2 else (np.nan, np.nan)
    _kpt_rows.append({
        'keypoint':   _id2name_c.get(str(_k), f'kpt_{_k}'),
        'n':          int(_m.sum()),
        'mean_score': float(_s.mean()),
        'mean_dist':  float(_d.mean()),
        'median_dist':float(np.median(_d)),
        'spearman_ρ': float(_rho),
        'p_value':    float(_pval),
    })

_kpt_df = (
    pd.DataFrame(_kpt_rows)
    .sort_values('mean_dist')
    .reset_index(drop=True)
)


def _rho_bg(series):
    """Green for ρ < 0 (higher score → lower error), red for ρ ≥ 0."""
    return [
        'background-color: #d4edda' if v < 0 else 'background-color: #f8d7da'
        for v in series
    ]


display(
    _kpt_df.style
    .apply(_rho_bg, subset=['spearman_ρ'])
    .format({
        'mean_score':  '{:.3f}',
        'mean_dist':   '{:.4f}',
        'median_dist': '{:.4f}',
        'spearman_ρ':  '{:+.3f}',
        'p_value':     '{:.2e}',
    }, na_rep='—')
    .set_caption(
        f'Per-keypoint calibration  |  {_model_label_c}  '
        f'|  ρ < 0 (green) = higher score → lower error  '
        f'|  dist unit: {dist_unit}'
    )
    .set_table_styles([{
        'selector': 'caption',
        'props': [('font-size', '13px'), ('font-weight', 'bold')],
    }])
)


## Score vs. keypoint error and reliability

Same calibration view as the cell above, for three prediction bundles side by side. Set the three paths in `CALIB_PRED_DIRS`.

The top row plots predicted keypoint score against keypoint error divided by the PCKh@0.5 threshold (0.5 × head size). The dashed line at 1 is that threshold. Color is log count, and the solid line is the mean normalized error in each score bin.

The bottom row is the reliability diagram: the fraction of joints correct under that same threshold. If a model's highest score is above 1, every score is divided by that maximum so the range sits in [0, 1]. The faint diagonal is perfect calibration and the shaded band is the per-bin gap.

In [ ]:
import json
import os.path as osp

import matplotlib.pyplot as plt
import numpy as np

# ── Configuration ─────────────────────────────────────────────────────────────
# Three prediction bundles, one per column. Each directory must contain
# manifest.json and frames.json.
#
# Example:
#   /local/home/nkoefarago/mmpose/benchmark/predictions/20260715_emdb_e2e/YOLO26-Pose-large
CALIB_PRED_DIRS = [
    '/local/home/nkoefarago/mmpose/benchmark/predictions/20260715_emdb_topdown/PCT-huge-rfdetr',
    '/local/home/nkoefarago/mmpose/benchmark/predictions/20260715_emdb_topdown/RTMPose-medium-rfdetr',
    '/local/home/nkoefarago/mmpose/benchmark/predictions/20260715_emdb_e2e/YOLO-Pose-large',
]
# Keypoints to include: list of names (str), list of indices (int), or None / 'all'.
CALIB_KEYPOINTS = None
# Plotted error is the pixel distance divided by the PCKh threshold
# (PCKH_THR × head size). A joint is correct when that ratio is below 1.
# Number of score buckets for the reliability diagram and the mean-error curve.
N_BINS = 20
# Bins with fewer samples than this threshold are hidden from the curves.
MIN_SAMPLES_PER_BIN = 10
# Upper percentile used for the error-envelope shading on the error plot.
ERROR_PERCENTILE = 75
# Ignore keypoint predictions below this score (skip near-background detections).
MIN_SCORE = 0.0
# Distance values above this percentile are clipped in the density plot only.
MAX_DIST_PERCENTILE = 99
# PCKh threshold. A joint is correct when its pixel error is strictly below
# PCKH_THR * head_size, matching mmpose `keypoint_pck_accuracy`.
PCKH_THR = 0.5
# Head-size fallbacks used only when the bundle has no head box and no
# labeled head joints (for example EMDB). 0.1 × bbox height approximates
# the MPII head size, 0.6 × head-box diagonal, on an 8-head body.
# Shoulder-to-hip is the fallback when the GT box is missing.
HEAD_SIZE_FROM_BODY_HEIGHT = 0.1
HEAD_SIZE_FROM_TORSO = 0.3


# ── Helpers ───────────────────────────────────────────────────────────────────
def _calib_load_bundle(pred_dir: str):
    for fname in ('manifest.json', 'frames.json'):
        p = osp.join(pred_dir, fname)
        if not osp.isfile(p):
            raise FileNotFoundError(f'Missing {fname} in {pred_dir}')
    with open(osp.join(pred_dir, 'manifest.json')) as f:
        manifest = json.load(f)
    with open(osp.join(pred_dir, 'frames.json')) as f:
        frames = json.load(f)
    return manifest, frames


def _calib_kpt_indices(kpts_cfg, dataset_meta: dict):
    """Return a list of keypoint indices, or None (= all keypoints)."""
    if kpts_cfg is None or kpts_cfg == 'all':
        return None
    id2name = dataset_meta.get('keypoint_id2name', {})
    name2id = {v: int(k) for k, v in id2name.items()}
    result = []
    for kpt in ([kpts_cfg] if not isinstance(kpts_cfg, (list, tuple)) else kpts_cfg):
        if isinstance(kpt, int):
            result.append(kpt)
        elif kpt in name2id:
            result.append(name2id[kpt])
        else:
            raise ValueError(f'Keypoint {kpt!r} not found. Available: {sorted(name2id)}')
    return result


def _calib_body_height(gt_inst):
    """Person height in pixels from a tight xyxy box, or None."""
    bbox = gt_inst.get('bbox')
    if bbox is None:
        return None
    b = np.asarray(bbox, dtype=np.float64).reshape(-1)
    if b.size < 4:
        return None
    height = abs(float(b[3] - b[1]))
    return height if height >= 1.0 else None


def _calib_head_size(gt_inst, name2id):
    """Head size (pixels) used as the PCKh normalizer.

    A joint is correct under PCKh@``PCKH_THR`` when its Euclidean error is
    strictly below ``PCKH_THR * head_size``. Resolution order:

    1. Annotated ``head_size`` (MPII already stores 0.6 × head-box diagonal).
    2. ``||head_top - head_bottom||`` when both joints are labeled.
    3. Ear-to-ear distance. That span is about one head width, which matches
       the scale of the MPII head size.
    4. Twice the inter-ocular distance (about half an ear-to-ear span).
    5. ``HEAD_SIZE_FROM_BODY_HEIGHT ×`` GT bbox height when the head is
       unlabeled. ``HEAD_SIZE_FROM_TORSO ×`` shoulder-to-hip length if the
       box is missing.

    Returns ``(head_size, rule)`` or ``(None, None)``.
    """
    stored = gt_inst.get('head_size')
    if stored is not None:
        hs = float(np.asarray(stored, dtype=np.float64).reshape(-1)[0])
        if hs >= 1.0:
            return hs, 'annotated'

    kpts = np.asarray(gt_inst.get('keypoints') or [], dtype=np.float64).reshape(-1, 2)
    vis = np.asarray(
        gt_inst.get('keypoints_visible') or [], dtype=np.float64).reshape(-1)

    def _xy(name):
        idx = name2id.get(name)
        if idx is None or idx >= len(kpts) or idx >= len(vis) or vis[idx] <= 0:
            return None
        return kpts[idx]

    def _span(a, b):
        if a is None or b is None:
            return None
        return float(np.linalg.norm(a - b))

    segment = _span(_xy('head_top'), _xy('head_bottom'))
    if segment is not None and segment >= 1.0:
        return segment, 'head_segment'

    ears = _span(_xy('left_ear'), _xy('right_ear'))
    if ears is not None and ears >= 1.0:
        return ears, 'ear_span'

    eyes = _span(_xy('left_eye'), _xy('right_eye'))
    if eyes is not None and 2.0 * eyes >= 1.0:
        return 2.0 * eyes, 'eye_span'

    body_h = _calib_body_height(gt_inst)
    if body_h is not None:
        hs = HEAD_SIZE_FROM_BODY_HEIGHT * body_h
        if hs >= 1.0:
            return hs, 'bbox_height'

    lsho, rsho = _xy('left_shoulder'), _xy('right_shoulder')
    lhip, rhip = _xy('left_hip'), _xy('right_hip')
    if all(pt is not None for pt in (lsho, rsho, lhip, rhip)):
        torso = float(np.linalg.norm(0.5 * (lsho + rsho) - 0.5 * (lhip + rhip)))
        hs = HEAD_SIZE_FROM_TORSO * torso
        if hs >= 1.0:
            return hs, 'torso'

    return None, None


def _calib_collect(frames, kpt_indices, min_score: float, name2id):
    """Collect (score, distance, kpt_id, pckh_correct) for matched joints.

    ``distance`` is the pixel error divided by the PCKh@``PCKH_THR`` threshold
    (``PCKH_THR * head_size``). A joint is correct when that ratio is below 1.
    Persons with no usable head size are skipped.
    """
    scores_out, dists_out, kpt_ids_out, correct_out = [], [], [], []
    head_rules = {}
    n_skipped_no_head = 0

    for frame in frames:
        matches = frame['metrics'].get('matches', [])
        pred_insts = frame['predictions']['instances']
        gt_insts   = frame['ground_truth']['instances']

        for match in matches:
            pred_inst = pred_insts[match['pred_idx']]
            gt_inst   = gt_insts[match['gt_idx']]
            head_size, rule = _calib_head_size(gt_inst, name2id)
            if head_size is None:
                n_skipped_no_head += 1
                continue
            head_rules[rule] = head_rules.get(rule, 0) + 1
            threshold = PCKH_THR * head_size

            pred_kpts   = pred_inst['keypoints']
            pred_scores = pred_inst.get('keypoint_scores', [1.0] * len(pred_kpts))
            gt_kpts     = gt_inst['keypoints']
            gt_vis      = gt_inst.get('keypoints_visible', [1.0] * len(gt_kpts))

            n_kpts  = min(len(pred_kpts), len(gt_kpts))
            indices = kpt_indices if kpt_indices is not None else range(n_kpts)

            for k in indices:
                if k >= n_kpts:
                    continue
                vis = float(gt_vis[k]) if k < len(gt_vis) else 1.0
                if vis == 0:
                    continue
                score = float(pred_scores[k]) if k < len(pred_scores) else 1.0
                if score < min_score:
                    continue
                dx = float(pred_kpts[k][0]) - float(gt_kpts[k][0])
                dy = float(pred_kpts[k][1]) - float(gt_kpts[k][1])
                dist_px = (dx * dx + dy * dy) ** 0.5
                scores_out.append(score)
                dists_out.append(dist_px / threshold)
                kpt_ids_out.append(int(k))
                correct_out.append(dist_px < threshold)

    return (
        np.array(scores_out, dtype=np.float32),
        np.array(dists_out,  dtype=np.float32),
        np.array(kpt_ids_out, dtype=np.int32),
        np.array(correct_out, dtype=bool),
        head_rules,
        n_skipped_no_head,
    )


def _calib_model_label(manifest: dict) -> str:
    """Readable name: model, capacity, and detector. Not the run-folder slug."""
    name = str(manifest.get('model_name') or 'model')
    variant = str(manifest.get('model_variant') or '')
    detector = detector_suffix_from_run_name(variant) if variant else None
    size = variant
    if detector and size.lower().endswith(detector.lower()):
        size = size[: -len(detector)].strip('-')
    label = name if not size else f'{name} {size}'
    if detector:
        label = f'{label} ({detector_display_name(detector)})'
    return label


def _calib_summarize(scores, dists, correct, label: str):
    """Bin one model's matches and return everything the two plots need."""
    if len(scores) == 0:
        raise ValueError(f'No (score, distance) pairs collected for {label}.')

    dist_clip = float(np.percentile(dists, MAX_DIST_PERCENTILE))
    dists_plot = np.clip(dists, 0.0, dist_clip)
    dist_unit = rf'error / (${PCKH_THR:g} \times$ head size)'

    # Heatmap and SimCC peaks are trained against a Gaussian of height 1 and are
    # not passed through a sigmoid, so the maximum can sit slightly above 1.
    # Divide every score by that maximum so the whole range lands in [0, 1].
    score_scale = float(scores.max())
    if score_scale > 1.0:
        scores = scores / score_scale
    else:
        score_scale = 1.0

    score_lo = float(max(MIN_SCORE, scores.min()))
    score_hi = float(scores.max())
    if score_hi <= score_lo:
        score_hi = score_lo + 1e-3
    score_pad = 0.02 * (score_hi - score_lo)
    score_xlim = (score_lo - score_pad, score_hi + score_pad)
    prob_scores = bool(float(scores.min()) >= -1e-3 and float(scores.max()) <= 1.01)

    bin_edges = np.linspace(score_lo, score_hi, N_BINS + 1)
    bin_centers = 0.5 * (bin_edges[:-1] + bin_edges[1:])
    bin_mean = np.full(N_BINS, np.nan)
    bin_p25 = np.full(N_BINS, np.nan)
    bin_p_hi = np.full(N_BINS, np.nan)
    bin_acc = np.full(N_BINS, np.nan)
    bin_conf = np.full(N_BINS, np.nan)
    bin_n = np.zeros(N_BINS, dtype=int)

    for i, (lo, hi) in enumerate(zip(bin_edges[:-1], bin_edges[1:])):
        mask = (scores >= lo) & (scores <= hi if i == N_BINS - 1 else scores < hi)
        n = int(mask.sum())
        bin_n[i] = n
        if n >= MIN_SAMPLES_PER_BIN:
            d = dists[mask]
            bin_mean[i] = d.mean()
            bin_p25[i] = np.percentile(d, 25)
            bin_p_hi[i] = np.percentile(d, ERROR_PERCENTILE)
            bin_acc[i] = float(correct[mask].mean())
            bin_conf[i] = float(scores[mask].mean())

    valid = ~np.isnan(bin_mean)
    ece = None
    if prob_scores and valid.any():
        w = bin_n[valid].astype(np.float64)
        ece = float(np.sum(w * np.abs(bin_acc[valid] - bin_conf[valid])) / w.sum())

    return {
        'label': label,
        'scores': scores,
        'dists': dists,
        'dists_plot': dists_plot,
        'dist_clip': dist_clip,
        'dist_unit': dist_unit,
        'score_scale': score_scale,
        'score_xlim': score_xlim,
        'prob_scores': prob_scores,
        'bin_edges': bin_edges,
        'bin_centers': bin_centers,
        'bin_mean': bin_mean,
        'bin_p25': bin_p25,
        'bin_p_hi': bin_p_hi,
        'bin_acc': bin_acc,
        'bin_conf': bin_conf,
        'bin_n': bin_n,
        'valid': valid,
        'pckh': float(correct.mean()),
        'ece': ece,
        'n': int(len(scores)),
    }


def _calib_bundle(pred_dir: str) -> dict:
    manifest, frames = _calib_load_bundle(pred_dir)
    dmeta = manifest.get('dataset_meta', {})
    kpt_idx = _calib_kpt_indices(CALIB_KEYPOINTS, dmeta)
    name2id = {v: int(k) for k, v in dmeta.get('keypoint_id2name', {}).items()}
    scores, dists, _kpt_ids, correct, head_rules, n_no_head = _calib_collect(
        frames, kpt_idx, MIN_SCORE, name2id)
    n_frames = len(frames)
    del frames

    label = _calib_model_label(manifest)
    if len(scores) == 0:
        raise ValueError(
            f'No (score, distance) pairs collected from {pred_dir}. '
            'Check matches / visibility / MIN_SCORE / head size.'
        )
    bundle = _calib_summarize(scores, dists, correct, label)
    bundle['n_frames'] = n_frames
    bundle['head_rules'] = head_rules
    bundle['n_no_head'] = n_no_head
    bundle['pred_dir'] = pred_dir
    return bundle


def _draw_score_vs_error(ax, bundle, *, show_ylabel=True):
    """Hexbin of score vs. keypoint error, with the per-bin mean overlaid."""
    hb = ax.hexbin(
        bundle['scores'], bundle['dists_plot'],
        gridsize=45, cmap='YlOrRd', mincnt=1, bins='log')
    valid = bundle['valid']
    centers = bundle['bin_centers']
    ax.axhline(
        1.0, color='0.15', ls='--', lw=1, alpha=0.8, zorder=4,
        label=latex_text(f'PCKh@{PCKH_THR:g}'))
    if valid.any():
        ax.plot(
            centers[valid], bundle['bin_mean'][valid],
            color='C0', linewidth=2, zorder=5,
            label=latex_text('mean error per bin'))
        ax.fill_between(
            centers[valid], bundle['bin_p25'][valid], bundle['bin_p_hi'][valid],
            color='C0', alpha=0.25,
            label=latex_text(f'25th–{ERROR_PERCENTILE}th pct.'))
    ax.set_xlabel(latex_text('predicted score'))
    if show_ylabel:
        ax.set_ylabel(latex_text(
            f'{bundle["dist_unit"]}  (clipped at {MAX_DIST_PERCENTILE}th pct.)'))
    ax.set_title(
        r'\textbf{' + latex_text(bundle['label']) + '}',
        fontsize=12, pad=6)
    ax.grid(True, alpha=0.3)
    handles, _labels = ax.get_legend_handles_labels()
    if handles:
        ax.legend(fontsize=8, loc='upper right')
    ax.set_xlim(*bundle['score_xlim'])
    ax.set_ylim(0, bundle['ylim'])
    return hb


def _draw_reliability(ax, bundle, *, show_ylabel=True, show_count_ylabel=True):
    """PCKh reliability diagram for one model."""
    ax_n = ax.twinx()
    edges = bundle['bin_edges']
    centers = bundle['bin_centers']
    bar_w = float(edges[1] - edges[0]) * 0.9
    ax_n.bar(
        centers, bundle['bin_n'], width=bar_w,
        color='C7', alpha=0.3, label=latex_text('sample count'))
    if show_count_ylabel:
        ax_n.set_ylabel(latex_text('# keypoints in bin'), color='C7', fontsize=9)
    ax_n.tick_params(axis='y', labelcolor='C7', labelsize=8)

    ax.plot(
        [0, 1], [0, 1], ls='--', color='k', lw=1, alpha=0.35,
        label=latex_text('perfect calibration'), zorder=2)
    valid = bundle['valid']
    if valid.any():
        order = np.argsort(bundle['bin_conf'][valid])
        rel_x = bundle['bin_conf'][valid][order]
        rel_y = bundle['bin_acc'][valid][order]
        ax.plot(
            rel_x, rel_y, 'o-', color='C2', linewidth=2, markersize=4,
            label=latex_text(f'PCKh@{PCKH_THR:g} accuracy'), zorder=5)
        if bundle['prob_scores']:
            ax.fill_between(
                rel_x, rel_y, rel_x, color='C3', alpha=0.25,
                label=latex_text('calibration gap'))

    if bundle['prob_scores']:
        ax.set_xlim(0, 1)
    else:
        ax.set_xlim(*bundle['score_xlim'])
    ax.set_ylim(0, 1)
    ax.set_xlabel(latex_text('mean predicted score'))
    if show_ylabel:
        ax.set_ylabel(latex_text(f'fraction correct (PCKh@{PCKH_THR:g})'))
    ax.grid(True, alpha=0.3)
    ax.tick_params(labelsize=8)

    lines, labels = ax.get_legend_handles_labels()
    bars, blabels = ax_n.get_legend_handles_labels()
    ax.legend(
        lines + bars, labels + blabels, fontsize=8, loc='upper left')

    note = f'PCKh@{PCKH_THR:g} = {bundle["pckh"]:.3f}'
    if bundle['ece'] is not None:
        note += f', ECE = {bundle["ece"]:.3f}'
    ax.text(
        0.02, 0.02, latex_text(note), transform=ax.transAxes,
        ha='left', va='bottom', fontsize=8,
        bbox=dict(boxstyle='round', facecolor='white', alpha=0.85, edgecolor='0.8'))


def _print_bundle(bundle):
    where = osp.join(
        osp.basename(osp.dirname(bundle['pred_dir'])),
        osp.basename(bundle['pred_dir']))
    scale_note = ''
    if bundle['score_scale'] > 1.0:
        scale_note = f' (divided by raw maximum {bundle["score_scale"]:.3f})'
    ece_note = ''
    if bundle['ece'] is not None:
        ece_note = f' | ECE {bundle["ece"]:.3f}'
    scores = bundle['scores']
    print(
        f'{bundle["label"]}  ({where})\n'
        f'  {bundle["n"]:,} keypoints from {bundle["n_frames"]} frames | '
        f'score [{scores.min():.3f}, {scores.max():.3f}]{scale_note} | '
        f'PCKh@{PCKH_THR:g} {bundle["pckh"]:.3f}{ece_note}'
    )
    if bundle['head_rules']:
        rule_txt = ', '.join(
            f'{k}={v:,}' for k, v in sorted(bundle['head_rules'].items()))
        print(f'  Head-size source (matched persons): {rule_txt}')
    if bundle['n_no_head']:
        print(f'  Skipped {bundle["n_no_head"]:,} matched persons with no usable head size.')
    if not bundle['prob_scores']:
        print('  Note: keypoint scores are outside [0, 1]; binning uses the observed range.')


# ── Load & plot ────────────────────────────────────────────────────────────────
if len(CALIB_PRED_DIRS) != 3 or any(not str(p).strip() for p in CALIB_PRED_DIRS):
    raise ValueError(
        'Set CALIB_PRED_DIRS to exactly 3 prediction directories '
        '(each containing manifest.json and frames.json).'
    )

bundles = [_calib_bundle(osp.abspath(p)) for p in CALIB_PRED_DIRS]
# One error scale for the top row. 1 is the PCKh threshold on every panel.
_shared_clip = max(b['dist_clip'] for b in bundles)
_shared_ylim = max(_shared_clip, 1.0) * 1.05
for bundle in bundles:
    bundle['dists_plot'] = np.clip(bundle['dists'], 0.0, _shared_clip)
    bundle['ylim'] = _shared_ylim
    _print_bundle(bundle)

# Colorbar sits in a narrow column beside each error plot. The matching empty
# cell under it keeps the reliability axes the same width.
fig = plt.figure(figsize=(17.2, 8.0))
outer = fig.add_gridspec(
    1, 3, wspace=0.18, left=0.055, right=0.975, top=0.93, bottom=0.08)
for col, bundle in enumerate(bundles):
    inner = outer[col].subgridspec(
        2, 2, width_ratios=[1.0, 0.065], wspace=0.06, hspace=0.22)
    ax_hex = fig.add_subplot(inner[0, 0])
    cax = fig.add_subplot(inner[0, 1])
    ax_rel = fig.add_subplot(inner[1, 0])
    hb = _draw_score_vs_error(ax_hex, bundle, show_ylabel=(col == 0))
    _draw_reliability(
        ax_rel, bundle,
        show_ylabel=(col == 0),
        show_count_ylabel=(col == 2))
    cb = fig.colorbar(hb, cax=cax)
    cb.set_label(latex_text(r'log$_{10}$(count)'), fontsize=8)
    cb.ax.tick_params(labelsize=7)

show_and_save_plot(fig, 'calibration_three_models', bbox_inches='tight')
